# Imports

In [1]:
import pandas as pd
import numpy as np
from tabulate import tabulate

from CortesAnalysisPackage import peakfitting as pfa
from CortesAnalysisPackage import componentfitting as ca

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats.mstats import linregress

import warnings

In [2]:
from scipy.optimize import least_squares

In [3]:
sims_dir = '../Simulation/'
DetectorReadings = pd.read_csv(sims_dir + 'DetectorReadings.csv', index_col=0)
sims = pd.read_csv(sims_dir + 'sims.csv', index_col=0)
sims.set_index('label', inplace=True)
soilinfos = pd.read_json(sims_dir + 'soilinfos.json')

In [4]:
si_ind = soilinfos['element_mat'].apply(lambda x: x[0]).loc['Si'].index(1)

In [5]:
sims['silicon_level'] = soilinfos['element_mat'].apply(lambda x: np.average(x, axis=0)[si_ind])

In [6]:
o_ind = soilinfos['element_mat'].apply(lambda x: x[0]).loc['O'].index(1)
sims['oxygen_level'] = soilinfos['element_mat'].apply(lambda x: np.average(x, axis=0)[o_ind])

In [7]:
sims['avg_density'] = soilinfos.density_mat.apply(lambda x: np.average(x))
elem_sims = sims[~sims['elem'].isna()]
elem_Readings = DetectorReadings[elem_sims.index]

In [8]:
material_sims = sims[sims['material'].notna()]
material_sims = material_sims[material_sims['material'] != 'Water']
material_sims = material_sims[(material_sims['avg_density'] < 1.4) & (material_sims['avg_density'] > 1.3)]

In [9]:
def activation_layer(target_val, true_carbon):
    if np.all(target_val == target_val[0]):
        print('all target values are the same')
        mean_val = np.mean(true_carbon)
        result = linregress([0, 1], [mean_val, mean_val])
        reg = lambda x: result.slope * x + result.intercept
        analyze = lambda x: np.clip(reg(x), 0, 1)
    else:
        print('target values are not the same')
        result = linregress(target_val, true_carbon)
        reg = lambda x: result.slope * x + result.intercept
        analyze = lambda x: np.clip(reg(x), 0, 1)
        # analyze = lambda x: reg(x)
    
    return result, analyze

In [10]:
def activation_layer_multivar(X, y):
    """
    Multivariate linear fit using SVD.
    X: shape (n_samples, n_features) or (n_features,) for 1D
    y: shape (n_samples,)
    Returns: (coeffs, analyze)
    """
    X = np.asarray(X)
    y = np.asarray(y)
    if X.ndim == 1:
        X = X.reshape(-1, 1)
    # Add bias column
    X_aug = np.hstack([X, np.ones((X.shape[0], 1))])
    # SVD solution to least squares: beta = pinv(X) @ y
    U, s, Vt = np.linalg.svd(X_aug, full_matrices=False)
    tol = 0  # or another small value
    s_inv = np.diag([1/x if x > tol else 0 for x in s])
    X_pinv = Vt.T @ s_inv @ U.T
    beta = X_pinv @ y  # beta[:-1]: weights, beta[-1]: intercept

    def reg(x):
        x = np.asarray(x)
        if x.ndim == 1:
            x = x.reshape(-1, X.shape[1])
        x_aug = np.hstack([x, np.ones((x.shape[0], 1))])
        return x_aug @ beta

    analyze = lambda x: np.clip(reg(x), 0, 1)
    return beta, analyze

In [11]:
def activation_layer_multivar_linreg(X, y):
    X = np.asarray(X)
    y = np.asarray(y)
    if X.ndim == 1:
        X = X.reshape(-1, 1)
    def reg(X, coeffs, intercept):
        return X @ coeffs + intercept
    def minimize_loss(params):
        coeffs = params[:-1]
        intercept = params[-1]
        y_pred = reg(X, coeffs, intercept)
        return (y - y_pred)
    
    # for _ in range(X.shape[1]):
    #     if np.all(X[:, _] == X[0, _]):
    #         print(f'feature {_} is constant across samples')
    #         mean_val = np.mean(y)
    #         X[:, _] = mean_val  # replace constant feature with mean of target to avoid singularity
    
    least_squares_result = least_squares(minimize_loss, x0=np.zeros(X.shape[1] + 1))
    beta = least_squares_result.x
    analyze = lambda x: np.clip(reg(x, beta[:-1], beta[-1]), 0, 1)

    # if np.all(X == X[0]):
    #     print('all target values are the same')
    #     mean_val = np.mean(y)
    #     result = linregress([0, 1], [mean_val, mean_val])
    #     reg = lambda x: result.slope * x + result.intercept
    #     analyze = lambda x: np.clip(reg(x), 0, 1)
    # else:
    #     print('target values are not the same')
    #     result = linregress(X.flatten(), y)
    #     reg = lambda x: result.slope * x + result.intercept
    #     analyze = lambda x: np.clip(reg(x), 0, 1)
    #     # analyze = lambda x: reg(x)
    
    return beta, analyze

# Peakfitting

In [12]:
def PF_Linear(training_df, test_df, training_exp_df, test_exp_df):
    """
    Perform peak fitting on the training and test data.
    """
    training_cols = training_df.columns
    test_cols = test_df.columns
    
    combined_exp_df = pd.concat([training_exp_df, test_exp_df], axis=0)
    combined_exp_df = combined_exp_df.loc[~combined_exp_df.index.duplicated(keep='first')]

    combined_df = pd.concat([training_df, test_df], axis=1)
    combined_df = combined_df.loc[:, ~combined_df.columns.duplicated(keep='first')]
    
    true_carbon = combined_exp_df['carbon_level'].values
    test_true_carbon = test_exp_df['carbon_level'].values
    training_true_carbon = training_exp_df['carbon_level'].values


    true_silicon = combined_exp_df['silicon_level'].values
    test_true_silicon = test_exp_df['silicon_level'].values
    training_true_silicon = training_exp_df['silicon_level'].values

    true_oxygen = combined_exp_df['oxygen_level'].values
    test_true_oxygen = test_exp_df['oxygen_level'].values
    training_true_oxygen = training_exp_df['oxygen_level'].values

    fitting_df, carbon_fitting_df, si_fitting_df, c_lines_df, si_lines_df = pfa.PeakFit(
        combined_df,
        bins=combined_df.index,
        c_window=[4.2, 4.7], 
        si_window=[1.5, 2],
        c_baseline='linear',
        si_baseline='linear',
        )
    


    fitting_df2, o_fitting_df, _fitting_df, o_lines_df, _lines_df = pfa.PeakFit(
        combined_df,
        bins=combined_df.index,
        c_window=[5.9, 6.3], 
        si_window=[1.5, 2],
        c_baseline='linear',
        si_baseline='linear',
        )
    
    fitting_df = fitting_df.set_index('label')
    training_fitting_df = fitting_df.loc[training_cols]
    test_fitting_df = fitting_df.loc[test_cols]
    
    carbon_peak_areas = fitting_df['Carbon Peak Area'].values
    training_carbon_peak_areas = training_fitting_df['Carbon Peak Area'].values
    test_carbon_peak_areas = test_fitting_df['Carbon Peak Area'].values
    
    silicon_peak_areas = fitting_df['Silicon Peak Area'].values
    training_silicon_peak_areas = training_fitting_df['Silicon Peak Area'].values
    test_silicon_peak_areas = test_fitting_df['Silicon Peak Area'].values
    
    fitting_df2 = fitting_df2.set_index('label')
    training_fitting_df2 = fitting_df2.loc[training_cols]
    test_fitting_df2 = fitting_df2.loc[test_cols]

    oxygen_peak_areas = fitting_df2['Carbon Peak Area'].values
    training_oxygen_peak_areas = training_fitting_df2['Carbon Peak Area'].values
    test_oxygen_peak_areas = test_fitting_df2['Carbon Peak Area'].values

    # result, analyze = activation_layer(training_carbon_peak_areas, training_true_carbon)
    result, analyze = activation_layer_multivar(np.array([training_carbon_peak_areas, training_silicon_peak_areas, training_oxygen_peak_areas]).T, training_true_carbon)

    predicted_carbon = analyze(np.array([carbon_peak_areas, silicon_peak_areas, oxygen_peak_areas]).T)
    predicted_training_carbon = analyze(np.array([training_carbon_peak_areas, training_silicon_peak_areas, training_oxygen_peak_areas]).T)
    predicted_test_carbon = analyze(np.array([test_carbon_peak_areas, test_silicon_peak_areas, test_oxygen_peak_areas]).T)

    mse_test = np.mean(np.abs(test_true_carbon - predicted_test_carbon))

    worst_accuracy = np.max(np.abs(test_true_carbon - predicted_test_carbon))


    info = {
        'mse': mse_test,
        'worst_accuracy': worst_accuracy,
        'fitting_df': fitting_df,
        'carbon_fitting_df': carbon_fitting_df,
        'si_fitting_df': si_fitting_df,
        'c_lines_df': c_lines_df,
        'si_lines_df': si_lines_df,
        'carbon_peak_areas': carbon_peak_areas,
        'silicon_peak_areas': silicon_peak_areas,
        'oxygen_peak_areas': oxygen_peak_areas,
        'training_carbon_peak_areas': training_carbon_peak_areas,
        'test_carbon_peak_areas': test_carbon_peak_areas,
        'true_carbon': true_carbon,
        'training_true_carbon': training_true_carbon,
        'test_true_carbon': test_true_carbon,
        'predicted_carbon': predicted_carbon,
        'predicted_training_carbon': predicted_training_carbon,
        'predicted_test_carbon': predicted_test_carbon,
        'method group': 'Peak Fitting',
    }
    return info

In [13]:
def PF_Exp_Falloff(training_df, test_df, training_exp_df, test_exp_df):
    """
    Perform peak fitting on the training and test data.
    """
    training_cols = training_df.columns
    test_cols = test_df.columns
    
    combined_exp_df = pd.concat([training_exp_df, test_exp_df], axis=0)
    combined_exp_df = combined_exp_df.loc[~combined_exp_df.index.duplicated(keep='first')]

    combined_df = pd.concat([training_df, test_df], axis=1)
    combined_df = combined_df.loc[:, ~combined_df.columns.duplicated(keep='first')]
    
    true_carbon = combined_exp_df['carbon_level'].values
    test_true_carbon = test_exp_df['carbon_level'].values
    training_true_carbon = training_exp_df['carbon_level'].values


    true_silicon = combined_exp_df['silicon_level'].values
    test_true_silicon = test_exp_df['silicon_level'].values
    training_true_silicon = training_exp_df['silicon_level'].values

    true_oxygen = combined_exp_df['oxygen_level'].values
    test_true_oxygen = test_exp_df['oxygen_level'].values
    training_true_oxygen = training_exp_df['oxygen_level'].values

    fitting_df, carbon_fitting_df, si_fitting_df, c_lines_df, si_lines_df = pfa.PeakFit(
        combined_df,
        bins=combined_df.index,
        c_window=[4.2, 4.7], 
        si_window=[1.675, 1.9],
        c_baseline='exp_falloff',
        si_baseline='exp_falloff',
        )
    


    fitting_df2, o_fitting_df, _fitting_df, o_lines_df, _lines_df = pfa.PeakFit(
        combined_df,
        bins=combined_df.index,
        c_window=[5.9, 6.3], 
        si_window=[1.675, 1.9],
        c_baseline='exp_falloff',
        si_baseline='exp_falloff',
        )
    
    fitting_df = fitting_df.set_index('label')
    training_fitting_df = fitting_df.loc[training_cols]
    test_fitting_df = fitting_df.loc[test_cols]
    
    carbon_peak_areas = fitting_df['Carbon Peak Area'].values
    training_carbon_peak_areas = training_fitting_df['Carbon Peak Area'].values
    test_carbon_peak_areas = test_fitting_df['Carbon Peak Area'].values
    
    silicon_peak_areas = fitting_df['Silicon Peak Area'].values
    training_silicon_peak_areas = training_fitting_df['Silicon Peak Area'].values
    test_silicon_peak_areas = test_fitting_df['Silicon Peak Area'].values
    
    fitting_df2 = fitting_df2.set_index('label')
    training_fitting_df2 = fitting_df2.loc[training_cols]
    test_fitting_df2 = fitting_df2.loc[test_cols]

    oxygen_peak_areas = fitting_df2['Carbon Peak Area'].values
    training_oxygen_peak_areas = training_fitting_df2['Carbon Peak Area'].values
    test_oxygen_peak_areas = test_fitting_df2['Carbon Peak Area'].values

    # result, analyze = activation_layer(training_carbon_peak_areas, training_true_carbon)
    result, analyze = activation_layer_multivar(np.array([
        training_carbon_peak_areas, 
        training_silicon_peak_areas, 
        training_oxygen_peak_areas
        ]).T, training_true_carbon)

    predicted_carbon = analyze(np.array([
        carbon_peak_areas, 
        silicon_peak_areas, 
        oxygen_peak_areas
        ]).T)
    predicted_training_carbon = analyze(np.array([
        training_carbon_peak_areas, 
        training_silicon_peak_areas, 
        training_oxygen_peak_areas
        ]).T)
    predicted_test_carbon = analyze(np.array([
        test_carbon_peak_areas, 
        test_silicon_peak_areas, 
        test_oxygen_peak_areas
    ]).T)

    mse_test = np.mean(np.abs(test_true_carbon - predicted_test_carbon))

    worst_accuracy = np.max(np.abs(test_true_carbon - predicted_test_carbon))

    info = {
        'mse': mse_test,
        'worst_accuracy': worst_accuracy,
        'fitting_df': fitting_df,
        'carbon_fitting_df': carbon_fitting_df,
        'si_fitting_df': si_fitting_df,
        'c_lines_df': c_lines_df,
        'si_lines_df': si_lines_df,
        'o_lines_df': o_lines_df,
        'carbon_peak_areas': carbon_peak_areas,
        'silicon_peak_areas': silicon_peak_areas,
        'oxygen_peak_areas': oxygen_peak_areas,
        'training_carbon_peak_areas': training_carbon_peak_areas,
        'test_carbon_peak_areas': test_carbon_peak_areas,
        'true_carbon': true_carbon,
        'true_silicon': true_silicon,
        'true_oxygen': true_oxygen,
        'training_true_carbon': training_true_carbon,
        'test_true_carbon': test_true_carbon,
        'predicted_carbon': predicted_carbon,
        'predicted_training_carbon': predicted_training_carbon,
        'predicted_test_carbon': predicted_test_carbon,
        'method group': 'Peak Fitting',
    }
    return info

In [14]:
def PF_Exp_Falloff_Full(training_df, test_df, training_exp_df, test_exp_df):
    """
    Perform peak fitting on the training and test data.
    """
    training_cols = training_df.columns
    test_cols = test_df.columns
    
    combined_exp_df = pd.concat([training_exp_df, test_exp_df], axis=0)
    combined_exp_df = combined_exp_df.loc[~combined_exp_df.index.duplicated(keep='first')]

    combined_df = pd.concat([training_df, test_df], axis=1)
    combined_df = combined_df.loc[:, ~combined_df.columns.duplicated(keep='first')]
    
    true_carbon = combined_exp_df['carbon_level'].values
    test_true_carbon = test_exp_df['carbon_level'].values
    training_true_carbon = training_exp_df['carbon_level'].values


    true_silicon = combined_exp_df['silicon_level'].values
    test_true_silicon = test_exp_df['silicon_level'].values
    training_true_silicon = training_exp_df['silicon_level'].values

    true_oxygen = combined_exp_df['oxygen_level'].values
    test_true_oxygen = test_exp_df['oxygen_level'].values
    training_true_oxygen = training_exp_df['oxygen_level'].values

    fitting_df, carbon_fitting_df, si_fitting_df, c_lines_df, si_lines_df = pfa.PeakFit(
        combined_df,
        bins=combined_df.index,
        c_window=[4.2, 4.7], 
        si_window=[1.675, 1.9],
        c_baseline='exp_falloff',
        si_baseline='exp_falloff',
        )
    


    fitting_df2, o_fitting_df, _fitting_df, o_lines_df, _lines_df = pfa.PeakFit(
        combined_df,
        bins=combined_df.index,
        c_window=[5.9, 6.3], 
        si_window=[1.675, 1.9],
        c_baseline='exp_falloff',
        si_baseline='exp_falloff',
        )
    
    fitting_df = fitting_df.set_index('label')
    training_fitting_df = fitting_df.loc[training_cols]
    test_fitting_df = fitting_df.loc[test_cols]
    
    carbon_peak_areas = fitting_df['Carbon Peak Area'].values
    training_carbon_peak_areas = training_fitting_df['Carbon Peak Area'].values
    test_carbon_peak_areas = test_fitting_df['Carbon Peak Area'].values
    
    silicon_peak_areas = fitting_df['Silicon Peak Area'].values
    training_silicon_peak_areas = training_fitting_df['Silicon Peak Area'].values
    test_silicon_peak_areas = test_fitting_df['Silicon Peak Area'].values
    
    fitting_df2 = fitting_df2.set_index('label')
    training_fitting_df2 = fitting_df2.loc[training_cols]
    test_fitting_df2 = fitting_df2.loc[test_cols]

    oxygen_peak_areas = fitting_df2['Carbon Peak Area'].values
    training_oxygen_peak_areas = training_fitting_df2['Carbon Peak Area'].values
    test_oxygen_peak_areas = test_fitting_df2['Carbon Peak Area'].values

    # result, analyze = activation_layer(training_carbon_peak_areas, training_true_carbon)
    result, analyze = activation_layer_multivar(np.array([
        training_carbon_peak_areas, 
        training_silicon_peak_areas, 
        training_oxygen_peak_areas
        ]).T, training_true_carbon)

    predicted_carbon = analyze(np.array([
        carbon_peak_areas, 
        silicon_peak_areas, 
        oxygen_peak_areas
        ]).T)
    predicted_training_carbon = analyze(np.array([
        training_carbon_peak_areas, 
        training_silicon_peak_areas, 
        training_oxygen_peak_areas
        ]).T)
    predicted_test_carbon = analyze(np.array([
        test_carbon_peak_areas, 
        test_silicon_peak_areas, 
        test_oxygen_peak_areas
    ]).T)

    mse_test = np.mean(np.abs(test_true_carbon - predicted_test_carbon))

    worst_accuracy = np.max(np.abs(test_true_carbon - predicted_test_carbon))

    info = {
        'mse': mse_test,
        'worst_accuracy': worst_accuracy,
        'fitting_df': fitting_df,
        'carbon_fitting_df': carbon_fitting_df,
        'si_fitting_df': si_fitting_df,
        'c_lines_df': c_lines_df,
        'si_lines_df': si_lines_df,
        'o_lines_df': o_lines_df,
        'carbon_peak_areas': carbon_peak_areas,
        'silicon_peak_areas': silicon_peak_areas,
        'oxygen_peak_areas': oxygen_peak_areas,
        'training_carbon_peak_areas': training_carbon_peak_areas,
        'test_carbon_peak_areas': test_carbon_peak_areas,
        'true_carbon': true_carbon,
        'true_silicon': true_silicon,
        'true_oxygen': true_oxygen,
        'training_true_carbon': training_true_carbon,
        'test_true_carbon': test_true_carbon,
        'predicted_carbon': predicted_carbon,
        'predicted_training_carbon': predicted_training_carbon,
        'predicted_test_carbon': predicted_test_carbon,
        'method group': 'Peak Fitting',
        'Variables': 'C Si and O'
    }
    return info

In [15]:
def PF_Exp_Falloff_Full_doublegauss(training_df, test_df, training_exp_df, test_exp_df):
    """
    Perform peak fitting on the training and test data.
    """
    training_cols = training_df.columns
    test_cols = test_df.columns
    
    combined_exp_df = pd.concat([training_exp_df, test_exp_df], axis=0)
    combined_exp_df = combined_exp_df.loc[~combined_exp_df.index.duplicated(keep='first')]

    combined_df = pd.concat([training_df, test_df], axis=1)
    combined_df = combined_df.loc[:, ~combined_df.columns.duplicated(keep='first')]
    
    true_carbon = combined_exp_df['carbon_level'].values
    test_true_carbon = test_exp_df['carbon_level'].values
    training_true_carbon = training_exp_df['carbon_level'].values


    true_silicon = combined_exp_df['silicon_level'].values
    test_true_silicon = test_exp_df['silicon_level'].values
    training_true_silicon = training_exp_df['silicon_level'].values

    true_oxygen = combined_exp_df['oxygen_level'].values
    test_true_oxygen = test_exp_df['oxygen_level'].values
    training_true_oxygen = training_exp_df['oxygen_level'].values

    fitting_df, carbon_fitting_df, si_fitting_df, c_lines_df, si_lines_df = pfa.PeakFit(
        combined_df,
        bins=combined_df.index,
        c_window=[4.2, 4.7], 
        si_window=[1.675, 1.9],
        c_peak='double_gauss',
        c_baseline='exp_falloff',
        si_baseline='exp_falloff',
        )
    


    fitting_df2, o_fitting_df, _fitting_df, o_lines_df, _lines_df = pfa.PeakFit(
        combined_df,
        bins=combined_df.index,
        c_window=[5.9, 6.3], 
        si_window=[1.675, 1.9],
        c_baseline='exp_falloff',
        si_baseline='exp_falloff',
        )
    
    fitting_df = fitting_df.set_index('label')
    training_fitting_df = fitting_df.loc[training_cols]
    test_fitting_df = fitting_df.loc[test_cols]
    
    carbon_peak_areas = fitting_df['Carbon Peak Area'].values
    training_carbon_peak_areas = training_fitting_df['Carbon Peak Area'].values
    test_carbon_peak_areas = test_fitting_df['Carbon Peak Area'].values
    
    silicon_peak_areas = fitting_df['Silicon Peak Area'].values
    training_silicon_peak_areas = training_fitting_df['Silicon Peak Area'].values
    test_silicon_peak_areas = test_fitting_df['Silicon Peak Area'].values
    
    fitting_df2 = fitting_df2.set_index('label')
    training_fitting_df2 = fitting_df2.loc[training_cols]
    test_fitting_df2 = fitting_df2.loc[test_cols]

    oxygen_peak_areas = fitting_df2['Carbon Peak Area'].values
    training_oxygen_peak_areas = training_fitting_df2['Carbon Peak Area'].values
    test_oxygen_peak_areas = test_fitting_df2['Carbon Peak Area'].values

    # result, analyze = activation_layer(training_carbon_peak_areas, training_true_carbon)
    result, analyze = activation_layer_multivar(np.array([
        training_carbon_peak_areas, 
        training_silicon_peak_areas, 
        training_oxygen_peak_areas
        ]).T, training_true_carbon)

    predicted_carbon = analyze(np.array([
        carbon_peak_areas, 
        silicon_peak_areas, 
        oxygen_peak_areas
        ]).T)
    predicted_training_carbon = analyze(np.array([
        training_carbon_peak_areas, 
        training_silicon_peak_areas, 
        training_oxygen_peak_areas
        ]).T)
    predicted_test_carbon = analyze(np.array([
        test_carbon_peak_areas, 
        test_silicon_peak_areas, 
        test_oxygen_peak_areas
    ]).T)

    mse_test = np.mean(np.abs(test_true_carbon - predicted_test_carbon))

    worst_accuracy = np.max(np.abs(test_true_carbon - predicted_test_carbon))

    info = {
        'mse': mse_test,
        'worst_accuracy': worst_accuracy,
        'fitting_df': fitting_df,
        'carbon_fitting_df': carbon_fitting_df,
        'si_fitting_df': si_fitting_df,
        'c_lines_df': c_lines_df,
        'si_lines_df': si_lines_df,
        'o_lines_df': o_lines_df,
        'carbon_peak_areas': carbon_peak_areas,
        'silicon_peak_areas': silicon_peak_areas,
        'oxygen_peak_areas': oxygen_peak_areas,
        'training_carbon_peak_areas': training_carbon_peak_areas,
        'test_carbon_peak_areas': test_carbon_peak_areas,
        'true_carbon': true_carbon,
        'true_silicon': true_silicon,
        'true_oxygen': true_oxygen,
        'training_true_carbon': training_true_carbon,
        'test_true_carbon': test_true_carbon,
        'predicted_carbon': predicted_carbon,
        'predicted_training_carbon': predicted_training_carbon,
        'predicted_test_carbon': predicted_test_carbon,
        'method group': 'Peak Fitting',
        'Variables': 'C Si and O'
    }
    return info

In [16]:
def PF_Exp_Falloff_Carbon_Silicon(training_df, test_df, training_exp_df, test_exp_df):
    """
    Perform peak fitting on the training and test data.
    """
    training_cols = training_df.columns
    test_cols = test_df.columns
    
    combined_exp_df = pd.concat([training_exp_df, test_exp_df], axis=0)
    combined_exp_df = combined_exp_df.loc[~combined_exp_df.index.duplicated(keep='first')]

    combined_df = pd.concat([training_df, test_df], axis=1)
    combined_df = combined_df.loc[:, ~combined_df.columns.duplicated(keep='first')]
    
    true_carbon = combined_exp_df['carbon_level'].values
    test_true_carbon = test_exp_df['carbon_level'].values
    training_true_carbon = training_exp_df['carbon_level'].values


    true_silicon = combined_exp_df['silicon_level'].values
    test_true_silicon = test_exp_df['silicon_level'].values
    training_true_silicon = training_exp_df['silicon_level'].values

    true_oxygen = combined_exp_df['oxygen_level'].values
    test_true_oxygen = test_exp_df['oxygen_level'].values
    training_true_oxygen = training_exp_df['oxygen_level'].values

    fitting_df, carbon_fitting_df, si_fitting_df, c_lines_df, si_lines_df = pfa.PeakFit(
        combined_df,
        bins=combined_df.index,
        c_window=[4.2, 4.7], 
        si_window=[1.675, 1.9],
        c_baseline='exp_falloff',
        si_baseline='exp_falloff',
        )
    


    fitting_df2, o_fitting_df, _fitting_df, o_lines_df, _lines_df = pfa.PeakFit(
        combined_df,
        bins=combined_df.index,
        c_window=[5.9, 6.3], 
        si_window=[1.675, 1.9],
        c_baseline='exp_falloff',
        si_baseline='exp_falloff',
        )
    
    fitting_df = fitting_df.set_index('label')
    training_fitting_df = fitting_df.loc[training_cols]
    test_fitting_df = fitting_df.loc[test_cols]
    
    carbon_peak_areas = fitting_df['Carbon Peak Area'].values
    training_carbon_peak_areas = training_fitting_df['Carbon Peak Area'].values
    test_carbon_peak_areas = test_fitting_df['Carbon Peak Area'].values
    
    silicon_peak_areas = fitting_df['Silicon Peak Area'].values
    training_silicon_peak_areas = training_fitting_df['Silicon Peak Area'].values
    test_silicon_peak_areas = test_fitting_df['Silicon Peak Area'].values
    
    fitting_df2 = fitting_df2.set_index('label')
    training_fitting_df2 = fitting_df2.loc[training_cols]
    test_fitting_df2 = fitting_df2.loc[test_cols]

    oxygen_peak_areas = fitting_df2['Carbon Peak Area'].values
    training_oxygen_peak_areas = training_fitting_df2['Carbon Peak Area'].values
    test_oxygen_peak_areas = test_fitting_df2['Carbon Peak Area'].values

    # result, analyze = activation_layer(training_carbon_peak_areas, training_true_carbon)
    result, analyze = activation_layer_multivar(np.array([
        training_carbon_peak_areas, 
        training_silicon_peak_areas, 
        # training_oxygen_peak_areas
        ]).T, training_true_carbon)

    predicted_carbon = analyze(np.array([
        carbon_peak_areas, 
        silicon_peak_areas, 
        # oxygen_peak_areas
        ]).T)
    predicted_training_carbon = analyze(np.array([
        training_carbon_peak_areas, 
        training_silicon_peak_areas, 
        # training_oxygen_peak_areas
        ]).T)
    predicted_test_carbon = analyze(np.array([
        test_carbon_peak_areas, 
        test_silicon_peak_areas, 
        # test_oxygen_peak_areas
    ]).T)

    mse_test = np.mean(np.abs(test_true_carbon - predicted_test_carbon))

    worst_accuracy = np.max(np.abs(test_true_carbon - predicted_test_carbon))

    info = {
        'mse': mse_test,
        'worst_accuracy': worst_accuracy,
        'fitting_df': fitting_df,
        'carbon_fitting_df': carbon_fitting_df,
        'si_fitting_df': si_fitting_df,
        'c_lines_df': c_lines_df,
        'si_lines_df': si_lines_df,
        'o_lines_df': o_lines_df,
        'carbon_peak_areas': carbon_peak_areas,
        'silicon_peak_areas': silicon_peak_areas,
        'oxygen_peak_areas': oxygen_peak_areas,
        'training_carbon_peak_areas': training_carbon_peak_areas,
        'test_carbon_peak_areas': test_carbon_peak_areas,
        'true_carbon': true_carbon,
        'true_silicon': true_silicon,
        'true_oxygen': true_oxygen,
        'training_true_carbon': training_true_carbon,
        'test_true_carbon': test_true_carbon,
        'predicted_carbon': predicted_carbon,
        'predicted_training_carbon': predicted_training_carbon,
        'predicted_test_carbon': predicted_test_carbon,
        'method group': 'Peak Fitting',
        'Variables': 'C and Si'
    }
    return info

In [17]:
def PF_Exp_Falloff_Carbon_Oxygen(training_df, test_df, training_exp_df, test_exp_df):
    """
    Perform peak fitting on the training and test data.
    """
    training_cols = training_df.columns
    test_cols = test_df.columns
    
    combined_exp_df = pd.concat([training_exp_df, test_exp_df], axis=0)
    combined_exp_df = combined_exp_df.loc[~combined_exp_df.index.duplicated(keep='first')]

    combined_df = pd.concat([training_df, test_df], axis=1)
    combined_df = combined_df.loc[:, ~combined_df.columns.duplicated(keep='first')]
    
    true_carbon = combined_exp_df['carbon_level'].values
    test_true_carbon = test_exp_df['carbon_level'].values
    training_true_carbon = training_exp_df['carbon_level'].values


    true_silicon = combined_exp_df['silicon_level'].values
    test_true_silicon = test_exp_df['silicon_level'].values
    training_true_silicon = training_exp_df['silicon_level'].values

    true_oxygen = combined_exp_df['oxygen_level'].values
    test_true_oxygen = test_exp_df['oxygen_level'].values
    training_true_oxygen = training_exp_df['oxygen_level'].values

    fitting_df, carbon_fitting_df, si_fitting_df, c_lines_df, si_lines_df = pfa.PeakFit(
        combined_df,
        bins=combined_df.index,
        c_window=[4.2, 4.7], 
        si_window=[1.675, 1.9],
        c_baseline='exp_falloff',
        si_baseline='exp_falloff',
        )
    


    fitting_df2, o_fitting_df, _fitting_df, o_lines_df, _lines_df = pfa.PeakFit(
        combined_df,
        bins=combined_df.index,
        c_window=[5.9, 6.3], 
        si_window=[1.675, 1.9],
        c_baseline='exp_falloff',
        si_baseline='exp_falloff',
        )
    
    fitting_df = fitting_df.set_index('label')
    training_fitting_df = fitting_df.loc[training_cols]
    test_fitting_df = fitting_df.loc[test_cols]
    
    carbon_peak_areas = fitting_df['Carbon Peak Area'].values
    training_carbon_peak_areas = training_fitting_df['Carbon Peak Area'].values
    test_carbon_peak_areas = test_fitting_df['Carbon Peak Area'].values
    
    silicon_peak_areas = fitting_df['Silicon Peak Area'].values
    training_silicon_peak_areas = training_fitting_df['Silicon Peak Area'].values
    test_silicon_peak_areas = test_fitting_df['Silicon Peak Area'].values
    
    fitting_df2 = fitting_df2.set_index('label')
    training_fitting_df2 = fitting_df2.loc[training_cols]
    test_fitting_df2 = fitting_df2.loc[test_cols]

    oxygen_peak_areas = fitting_df2['Carbon Peak Area'].values
    training_oxygen_peak_areas = training_fitting_df2['Carbon Peak Area'].values
    test_oxygen_peak_areas = test_fitting_df2['Carbon Peak Area'].values

    # result, analyze = activation_layer(training_carbon_peak_areas, training_true_carbon)
    result, analyze = activation_layer_multivar(np.array([
        training_carbon_peak_areas, 
        # training_silicon_peak_areas, 
        training_oxygen_peak_areas
        ]).T, training_true_carbon)

    predicted_carbon = analyze(np.array([
        carbon_peak_areas, 
        # silicon_peak_areas, 
        oxygen_peak_areas
        ]).T)
    predicted_training_carbon = analyze(np.array([
        training_carbon_peak_areas, 
        # training_silicon_peak_areas, 
        training_oxygen_peak_areas
        ]).T)
    predicted_test_carbon = analyze(np.array([
        test_carbon_peak_areas, 
        # test_silicon_peak_areas, 
        test_oxygen_peak_areas
    ]).T)

    mse_test = np.mean(np.abs(test_true_carbon - predicted_test_carbon))

    worst_accuracy = np.max(np.abs(test_true_carbon - predicted_test_carbon))

    info = {
        'mse': mse_test,
        'worst_accuracy': worst_accuracy,
        'fitting_df': fitting_df,
        'carbon_fitting_df': carbon_fitting_df,
        'si_fitting_df': si_fitting_df,
        'c_lines_df': c_lines_df,
        'si_lines_df': si_lines_df,
        'o_lines_df': o_lines_df,
        'carbon_peak_areas': carbon_peak_areas,
        'silicon_peak_areas': silicon_peak_areas,
        'oxygen_peak_areas': oxygen_peak_areas,
        'training_carbon_peak_areas': training_carbon_peak_areas,
        'test_carbon_peak_areas': test_carbon_peak_areas,
        'true_carbon': true_carbon,
        'true_silicon': true_silicon,
        'true_oxygen': true_oxygen,
        'training_true_carbon': training_true_carbon,
        'test_true_carbon': test_true_carbon,
        'predicted_carbon': predicted_carbon,
        'predicted_training_carbon': predicted_training_carbon,
        'predicted_test_carbon': predicted_test_carbon,
        'method group': 'Peak Fitting',
        'Variables': 'C and O'
    }
    return info

In [18]:
def PF_Exp_Falloff_Carbon_Only(training_df, test_df, training_exp_df, test_exp_df):
    """
    Perform peak fitting on the training and test data.
    """
    training_cols = training_df.columns
    test_cols = test_df.columns
    
    combined_exp_df = pd.concat([training_exp_df, test_exp_df], axis=0)
    combined_exp_df = combined_exp_df.loc[~combined_exp_df.index.duplicated(keep='first')]

    combined_df = pd.concat([training_df, test_df], axis=1)
    combined_df = combined_df.loc[:, ~combined_df.columns.duplicated(keep='first')]
    
    true_carbon = combined_exp_df['carbon_level'].values
    test_true_carbon = test_exp_df['carbon_level'].values
    training_true_carbon = training_exp_df['carbon_level'].values


    true_silicon = combined_exp_df['silicon_level'].values
    test_true_silicon = test_exp_df['silicon_level'].values
    training_true_silicon = training_exp_df['silicon_level'].values

    true_oxygen = combined_exp_df['oxygen_level'].values
    test_true_oxygen = test_exp_df['oxygen_level'].values
    training_true_oxygen = training_exp_df['oxygen_level'].values

    fitting_df, carbon_fitting_df, si_fitting_df, c_lines_df, si_lines_df = pfa.PeakFit(
        combined_df,
        bins=combined_df.index,
        c_window=[4.2, 4.7], 
        si_window=[1.675, 1.9],
        c_baseline='exp_falloff',
        si_baseline='exp_falloff',
        )
    


    fitting_df2, o_fitting_df, _fitting_df, o_lines_df, _lines_df = pfa.PeakFit(
        combined_df,
        bins=combined_df.index,
        c_window=[5.9, 6.3], 
        si_window=[1.675, 1.9],
        c_baseline='exp_falloff',
        si_baseline='exp_falloff',
        )
    
    fitting_df = fitting_df.set_index('label')
    training_fitting_df = fitting_df.loc[training_cols]
    test_fitting_df = fitting_df.loc[test_cols]
    
    carbon_peak_areas = fitting_df['Carbon Peak Area'].values
    training_carbon_peak_areas = training_fitting_df['Carbon Peak Area'].values
    test_carbon_peak_areas = test_fitting_df['Carbon Peak Area'].values
    
    silicon_peak_areas = fitting_df['Silicon Peak Area'].values
    training_silicon_peak_areas = training_fitting_df['Silicon Peak Area'].values
    test_silicon_peak_areas = test_fitting_df['Silicon Peak Area'].values
    
    fitting_df2 = fitting_df2.set_index('label')
    training_fitting_df2 = fitting_df2.loc[training_cols]
    test_fitting_df2 = fitting_df2.loc[test_cols]

    oxygen_peak_areas = fitting_df2['Carbon Peak Area'].values
    training_oxygen_peak_areas = training_fitting_df2['Carbon Peak Area'].values
    test_oxygen_peak_areas = test_fitting_df2['Carbon Peak Area'].values

    # result, analyze = activation_layer(training_carbon_peak_areas, training_true_carbon)
    result, analyze = activation_layer_multivar(np.array([
        training_carbon_peak_areas, 
        # training_silicon_peak_areas, 
        # training_oxygen_peak_areas
        ]).T, training_true_carbon)

    predicted_carbon = analyze(np.array([
        carbon_peak_areas, 
        # silicon_peak_areas, 
        # oxygen_peak_areas
        ]).T)
    predicted_training_carbon = analyze(np.array([
        training_carbon_peak_areas, 
        # training_silicon_peak_areas, 
        # training_oxygen_peak_areas
        ]).T)
    predicted_test_carbon = analyze(np.array([
        test_carbon_peak_areas, 
        # test_silicon_peak_areas, 
        # test_oxygen_peak_areas
    ]).T)

    mse_test = np.mean(np.abs(test_true_carbon - predicted_test_carbon))

    worst_accuracy = np.max(np.abs(test_true_carbon - predicted_test_carbon))

    info = {
        'mse': mse_test,
        'worst_accuracy': worst_accuracy,
        'fitting_df': fitting_df,
        'carbon_fitting_df': carbon_fitting_df,
        'si_fitting_df': si_fitting_df,
        'c_lines_df': c_lines_df,
        'si_lines_df': si_lines_df,
        'o_lines_df': o_lines_df,
        'carbon_peak_areas': carbon_peak_areas,
        'silicon_peak_areas': silicon_peak_areas,
        'oxygen_peak_areas': oxygen_peak_areas,
        'training_carbon_peak_areas': training_carbon_peak_areas,
        'test_carbon_peak_areas': test_carbon_peak_areas,
        'true_carbon': true_carbon,
        'true_silicon': true_silicon,
        'true_oxygen': true_oxygen,
        'training_true_carbon': training_true_carbon,
        'test_true_carbon': test_true_carbon,
        'predicted_carbon': predicted_carbon,
        'predicted_training_carbon': predicted_training_carbon,
        'predicted_test_carbon': predicted_test_carbon,
        'method group': 'Peak Fitting',
        'Variables': 'C only'
    }
    return info

# Edgemember fitting

In [19]:
# set na to 0 for carbon_level in elem_sims
elem_sims['carbon_level'] = elem_sims['carbon_level'].fillna(0)

/tmp/ipykernel_3363569/3888097818.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  elem_sims['carbon_level'] = elem_sims['carbon_level'].fillna(0)


In [20]:
elem_sims.at['C', 'carbon_level'] = 1.0

In [21]:
def CAelem(training_df, test_df, training_exp_df, test_exp_df, elem_df=elem_Readings, elem_exp_df=elem_sims):
    """
    Perform component analysis on the training and test data.
    """
    training_cols = training_df.columns
    test_cols = test_df.columns
    
    combined_df = pd.concat([training_df, test_df], axis=1)
    combined_df = combined_df.loc[:, ~combined_df.columns.duplicated()]
    combined_exp_df = pd.concat([training_exp_df, test_exp_df], axis=0)
    combined_exp_df = combined_exp_df.loc[~combined_exp_df.index.duplicated(keep='first')]

    true_carbon = combined_exp_df['carbon_level'].values
    test_true_carbon = test_exp_df['carbon_level'].values
    training_true_carbon = training_exp_df['carbon_level'].values
    
    true_silicon = combined_exp_df['silicon_level'].values
    test_true_silicon = test_exp_df['silicon_level'].values
    training_true_silicon = training_exp_df['silicon_level'].values

    true_oxygen = combined_exp_df['oxygen_level'].values
    test_true_oxygen = test_exp_df['oxygen_level'].values
    training_true_oxygen = training_exp_df['oxygen_level'].values

    elem_df = elem_df.loc[combined_df.index]
    _combined_df = pd.concat([combined_df, elem_df], axis=1)
    _combined_exp_df = pd.concat([combined_exp_df, elem_exp_df], axis=0)
    
    fitting_df, predicted_df = ca.Analyze(
        _combined_df,
        bins=combined_df.index,
        train_cols=list(elem_exp_df.index),
        true_c_concentrations=_combined_exp_df['carbon_level'].values,
        normalize=False,
        window = [1.0, 7]
    )
    
    carbon_portions = fitting_df['C'].loc[combined_df.columns].values
    training_carbon_portions = fitting_df['C'].loc[training_cols].values
    test_carbon_portions = fitting_df['C'].loc[test_cols].values

    silicon_portions = fitting_df['Si'].loc[combined_df.columns].values
    training_silicon_portions = fitting_df['Si'].loc[training_cols].values
    test_silicon_portions = fitting_df['Si'].loc[test_cols].values

    oxygen_portions = fitting_df['O'].loc[combined_df.columns].values
    training_oxygen_portions = fitting_df['O'].loc[training_cols].values
    test_oxygen_portions = fitting_df['O'].loc[test_cols].values

    result, analyze = activation_layer_multivar(np.array([training_carbon_portions, training_silicon_portions, training_oxygen_portions]).T, np.array(training_true_carbon))

    predicted_carbon = analyze(np.array([fitting_df['C'].loc[combined_df.columns].values, fitting_df['Si'].loc[combined_df.columns].values, fitting_df['O'].loc[combined_df.columns].values]).T)
    predicted_training_carbon = analyze(np.array([fitting_df['C'].loc[training_cols].values, fitting_df['Si'].loc[training_cols].values, fitting_df['O'].loc[training_cols].values]).T)
    predicted_test_carbon = analyze(np.array([fitting_df['C'].loc[test_cols].values, fitting_df['Si'].loc[test_cols].values, fitting_df['O'].loc[test_cols].values]).T)

    mse_test = np.mean(np.abs(test_true_carbon - predicted_test_carbon))

    worst_accuracy = np.max(np.abs(test_true_carbon - predicted_test_carbon))

    info = {
        'mse': mse_test,
        'worst_accuracy': worst_accuracy,
        'fitting_df': fitting_df,
        'predicted_df': predicted_df,
        'combined_df': _combined_df,
        'train_cols': list(elem_exp_df.index),
        'carbon_portions': carbon_portions,
        'silicon_portions': silicon_portions,
        'oxygen_portions': oxygen_portions,
        'training_carbon_portions': training_carbon_portions,
        'test_carbon_portions': test_carbon_portions,
        'true_carbon': true_carbon,
        'true_silicon': true_silicon,
        'true_oxygen': true_oxygen,
        'training_true_carbon': training_true_carbon,
        'test_true_carbon': test_true_carbon,
        'predicted_carbon': predicted_carbon,
        'predicted_training_carbon': predicted_training_carbon,
        'predicted_test_carbon': predicted_test_carbon,
        'method group': 'Component Analysis',
        
    }
    return info

In [22]:
def CAelemConvex(training_df, test_df, training_exp_df, test_exp_df, elem_df=elem_Readings, elem_exp_df=elem_sims):
    """
    Perform component analysis on the training and test data.
    """
    training_cols = training_df.columns
    test_cols = test_df.columns
    
    combined_df = pd.concat([training_df, test_df], axis=1)
    combined_df = combined_df.loc[:, ~combined_df.columns.duplicated()]
    combined_exp_df = pd.concat([training_exp_df, test_exp_df], axis=0)
    combined_exp_df = combined_exp_df.loc[~combined_exp_df.index.duplicated(keep='first')]

    true_carbon = combined_exp_df['carbon_level'].values
    test_true_carbon = test_exp_df['carbon_level'].values
    training_true_carbon = training_exp_df['carbon_level'].values
    
    true_silicon = combined_exp_df['silicon_level'].values
    test_true_silicon = test_exp_df['silicon_level'].values
    training_true_silicon = training_exp_df['silicon_level'].values

    true_oxygen = combined_exp_df['oxygen_level'].values
    test_true_oxygen = test_exp_df['oxygen_level'].values
    training_true_oxygen = training_exp_df['oxygen_level'].values

    elem_df = elem_df.loc[combined_df.index]
    _combined_df = pd.concat([combined_df, elem_df], axis=1)
    _combined_exp_df = pd.concat([combined_exp_df, elem_exp_df], axis=0)
    
    fitting_df, predicted_df = ca.Analyze(
        _combined_df,
        bins=combined_df.index,
        train_cols=list(elem_exp_df.index),
        true_c_concentrations=_combined_exp_df['carbon_level'].values,
        normalize=False,
        window = [1.0, 7],
        minimizer='convex'
    )
    
    carbon_portions = fitting_df['C'].loc[combined_df.columns].values
    training_carbon_portions = fitting_df['C'].loc[training_cols].values
    test_carbon_portions = fitting_df['C'].loc[test_cols].values

    silicon_portions = fitting_df['Si'].loc[combined_df.columns].values
    training_silicon_portions = fitting_df['Si'].loc[training_cols].values
    test_silicon_portions = fitting_df['Si'].loc[test_cols].values

    oxygen_portions = fitting_df['O'].loc[combined_df.columns].values
    training_oxygen_portions = fitting_df['O'].loc[training_cols].values
    test_oxygen_portions = fitting_df['O'].loc[test_cols].values

    result, analyze = activation_layer_multivar(np.array([training_carbon_portions, training_silicon_portions, training_oxygen_portions]).T, np.array(training_true_carbon))

    predicted_carbon = analyze(np.array([fitting_df['C'].loc[combined_df.columns].values, fitting_df['Si'].loc[combined_df.columns].values, fitting_df['O'].loc[combined_df.columns].values]).T)
    predicted_training_carbon = analyze(np.array([fitting_df['C'].loc[training_cols].values, fitting_df['Si'].loc[training_cols].values, fitting_df['O'].loc[training_cols].values]).T)
    predicted_test_carbon = analyze(np.array([fitting_df['C'].loc[test_cols].values, fitting_df['Si'].loc[test_cols].values, fitting_df['O'].loc[test_cols].values]).T)

    mse_test = np.mean(np.abs(test_true_carbon - predicted_test_carbon))

    worst_accuracy = np.max(np.abs(test_true_carbon - predicted_test_carbon))

    info = {
        'mse': mse_test,
        'worst_accuracy': worst_accuracy,
        'fitting_df': fitting_df,
        'predicted_df': predicted_df,
        'combined_df': _combined_df,
        'train_cols': list(elem_exp_df.index),
        'carbon_portions': carbon_portions,
        'silicon_portions': silicon_portions,
        'oxygen_portions': oxygen_portions,
        'training_carbon_portions': training_carbon_portions,
        'test_carbon_portions': test_carbon_portions,
        'true_carbon': true_carbon,
        'true_silicon': true_silicon,
        'true_oxygen': true_oxygen,
        'training_true_carbon': training_true_carbon,
        'test_true_carbon': test_true_carbon,
        'predicted_carbon': predicted_carbon,
        'predicted_training_carbon': predicted_training_carbon,
        'predicted_test_carbon': predicted_test_carbon,
        'method group': 'Component Analysis',
        
    }
    return info

In [23]:
def CAelemFull(training_df, test_df, training_exp_df, test_exp_df, elem_df=elem_Readings, elem_exp_df=elem_sims):
    """
    Perform component analysis on the training and test data.
    """
    training_cols = training_df.columns
    test_cols = test_df.columns
    
    combined_df = pd.concat([training_df, test_df], axis=1)
    combined_df = combined_df.loc[:, ~combined_df.columns.duplicated()]
    combined_exp_df = pd.concat([training_exp_df, test_exp_df], axis=0)
    combined_exp_df = combined_exp_df.loc[~combined_exp_df.index.duplicated(keep='first')]

    true_carbon = combined_exp_df['carbon_level'].values
    test_true_carbon = test_exp_df['carbon_level'].values
    training_true_carbon = training_exp_df['carbon_level'].values
    
    true_silicon = combined_exp_df['silicon_level'].values
    test_true_silicon = test_exp_df['silicon_level'].values
    training_true_silicon = training_exp_df['silicon_level'].values

    true_oxygen = combined_exp_df['oxygen_level'].values
    test_true_oxygen = test_exp_df['oxygen_level'].values
    training_true_oxygen = training_exp_df['oxygen_level'].values

    elem_df = elem_df.loc[combined_df.index]
    _combined_df = pd.concat([combined_df, elem_df], axis=1)
    _combined_exp_df = pd.concat([combined_exp_df, elem_exp_df], axis=0)
    
    fitting_df, predicted_df = ca.Analyze(
        _combined_df,
        bins=combined_df.index,
        train_cols=list(elem_exp_df.index),
        true_c_concentrations=_combined_exp_df['carbon_level'].values,
        normalize=False,
        window = [1.0, 7]
    )
    
    carbon_portions = fitting_df['C'].loc[combined_df.columns].values
    training_carbon_portions = fitting_df['C'].loc[training_cols].values
    test_carbon_portions = fitting_df['C'].loc[test_cols].values

    silicon_portions = fitting_df['Si'].loc[combined_df.columns].values
    training_silicon_portions = fitting_df['Si'].loc[training_cols].values
    test_silicon_portions = fitting_df['Si'].loc[test_cols].values

    oxygen_portions = fitting_df['O'].loc[combined_df.columns].values
    training_oxygen_portions = fitting_df['O'].loc[training_cols].values
    test_oxygen_portions = fitting_df['O'].loc[test_cols].values

    result, analyze = activation_layer_multivar(np.array([training_carbon_portions, training_silicon_portions, training_oxygen_portions]).T, np.array(training_true_carbon))

    predicted_carbon = analyze(np.array([fitting_df['C'].loc[combined_df.columns].values, fitting_df['Si'].loc[combined_df.columns].values, fitting_df['O'].loc[combined_df.columns].values]).T)
    predicted_training_carbon = analyze(np.array([fitting_df['C'].loc[training_cols].values, fitting_df['Si'].loc[training_cols].values, fitting_df['O'].loc[training_cols].values]).T)
    predicted_test_carbon = analyze(np.array([fitting_df['C'].loc[test_cols].values, fitting_df['Si'].loc[test_cols].values, fitting_df['O'].loc[test_cols].values]).T)

    mse_test = np.mean(np.abs(test_true_carbon - predicted_test_carbon))

    worst_accuracy = np.max(np.abs(test_true_carbon - predicted_test_carbon))

    info = {
        'mse': mse_test,
        'worst_accuracy': worst_accuracy,
        'fitting_df': fitting_df,
        'predicted_df': predicted_df,
        'combined_df': _combined_df,
        'train_cols': list(elem_exp_df.index),
        'carbon_portions': carbon_portions,
        'silicon_portions': silicon_portions,
        'oxygen_portions': oxygen_portions,
        'training_carbon_portions': training_carbon_portions,
        'test_carbon_portions': test_carbon_portions,
        'true_carbon': true_carbon,
        'true_silicon': true_silicon,
        'true_oxygen': true_oxygen,
        'training_true_carbon': training_true_carbon,
        'test_true_carbon': test_true_carbon,
        'predicted_carbon': predicted_carbon,
        'predicted_training_carbon': predicted_training_carbon,
        'predicted_test_carbon': predicted_test_carbon,
        'method group': 'Component Analysis',
        'Variables': 'C Si and O'
        
    }
    return info

In [24]:
def CAelemCSi(training_df, test_df, training_exp_df, test_exp_df, elem_df=elem_Readings, elem_exp_df=elem_sims):
    """
    Perform component analysis on the training and test data.
    """
    training_cols = training_df.columns
    test_cols = test_df.columns
    
    combined_df = pd.concat([training_df, test_df], axis=1)
    combined_df = combined_df.loc[:, ~combined_df.columns.duplicated()]
    combined_exp_df = pd.concat([training_exp_df, test_exp_df], axis=0)
    combined_exp_df = combined_exp_df.loc[~combined_exp_df.index.duplicated(keep='first')]

    true_carbon = combined_exp_df['carbon_level'].values
    test_true_carbon = test_exp_df['carbon_level'].values
    training_true_carbon = training_exp_df['carbon_level'].values
    
    true_silicon = combined_exp_df['silicon_level'].values
    test_true_silicon = test_exp_df['silicon_level'].values
    training_true_silicon = training_exp_df['silicon_level'].values

    true_oxygen = combined_exp_df['oxygen_level'].values
    test_true_oxygen = test_exp_df['oxygen_level'].values
    training_true_oxygen = training_exp_df['oxygen_level'].values

    elem_df = elem_df.loc[combined_df.index]
    _combined_df = pd.concat([combined_df, elem_df], axis=1)
    _combined_exp_df = pd.concat([combined_exp_df, elem_exp_df], axis=0)
    
    fitting_df, predicted_df = ca.Analyze(
        _combined_df,
        bins=combined_df.index,
        train_cols=list(elem_exp_df.index),
        true_c_concentrations=_combined_exp_df['carbon_level'].values,
        normalize=False,
        window = [1.0, 7]
    )
    
    carbon_portions = fitting_df['C'].loc[combined_df.columns].values
    training_carbon_portions = fitting_df['C'].loc[training_cols].values
    test_carbon_portions = fitting_df['C'].loc[test_cols].values

    silicon_portions = fitting_df['Si'].loc[combined_df.columns].values
    training_silicon_portions = fitting_df['Si'].loc[training_cols].values
    test_silicon_portions = fitting_df['Si'].loc[test_cols].values

    oxygen_portions = fitting_df['O'].loc[combined_df.columns].values
    training_oxygen_portions = fitting_df['O'].loc[training_cols].values
    test_oxygen_portions = fitting_df['O'].loc[test_cols].values

    result, analyze = activation_layer_multivar(np.array([
        training_carbon_portions, 
        training_silicon_portions, 
        # training_oxygen_portions
        ]).T, np.array(training_true_carbon))

    predicted_carbon = analyze(np.array([
        carbon_portions, 
        silicon_portions, 
        # oxygen_portions
        ]).T)
    predicted_training_carbon = analyze(np.array([
        training_carbon_portions, 
        training_silicon_portions, 
        # training_oxygen_portions
        ]).T)
    predicted_test_carbon = analyze(np.array([
        test_carbon_portions, 
        test_silicon_portions, 
        # test_oxygen_portions
        ]).T)

    mse_test = np.mean(np.abs(test_true_carbon - predicted_test_carbon))
    worst_accuracy = np.max(np.abs(test_true_carbon - predicted_test_carbon))

    info = {
        'mse': mse_test,
        'worst_accuracy': worst_accuracy,
        'fitting_df': fitting_df,
        'predicted_df': predicted_df,
        'combined_df': _combined_df,
        'train_cols': list(elem_exp_df.index),
        'carbon_portions': carbon_portions,
        'silicon_portions': silicon_portions,
        'oxygen_portions': oxygen_portions,
        'training_carbon_portions': training_carbon_portions,
        'test_carbon_portions': test_carbon_portions,
        'true_carbon': true_carbon,
        'true_silicon': true_silicon,
        'true_oxygen': true_oxygen,
        'training_true_carbon': training_true_carbon,
        'test_true_carbon': test_true_carbon,
        'predicted_carbon': predicted_carbon,
        'predicted_training_carbon': predicted_training_carbon,
        'predicted_test_carbon': predicted_test_carbon,
        'method group': 'Component Analysis',
        'Variables': 'C and Si'
        
    }
    return info

In [25]:
def CAelemC(training_df, test_df, training_exp_df, test_exp_df, elem_df=elem_Readings, elem_exp_df=elem_sims):
    """
    Perform component analysis on the training and test data.
    """
    training_cols = training_df.columns
    test_cols = test_df.columns
    
    combined_df = pd.concat([training_df, test_df], axis=1)
    combined_df = combined_df.loc[:, ~combined_df.columns.duplicated()]
    combined_exp_df = pd.concat([training_exp_df, test_exp_df], axis=0)
    combined_exp_df = combined_exp_df.loc[~combined_exp_df.index.duplicated(keep='first')]

    true_carbon = combined_exp_df['carbon_level'].values
    test_true_carbon = test_exp_df['carbon_level'].values
    training_true_carbon = training_exp_df['carbon_level'].values
    
    true_silicon = combined_exp_df['silicon_level'].values
    test_true_silicon = test_exp_df['silicon_level'].values
    training_true_silicon = training_exp_df['silicon_level'].values

    true_oxygen = combined_exp_df['oxygen_level'].values
    test_true_oxygen = test_exp_df['oxygen_level'].values
    training_true_oxygen = training_exp_df['oxygen_level'].values

    elem_df = elem_df.loc[combined_df.index]
    _combined_df = pd.concat([combined_df, elem_df], axis=1)
    _combined_exp_df = pd.concat([combined_exp_df, elem_exp_df], axis=0)
    
    fitting_df, predicted_df = ca.Analyze(
        _combined_df,
        bins=combined_df.index,
        train_cols=list(elem_exp_df.index),
        true_c_concentrations=_combined_exp_df['carbon_level'].values,
        normalize=True,
        window = [1.0, 7]
    )
    
    carbon_portions = fitting_df['C'].loc[combined_df.columns].values
    training_carbon_portions = fitting_df['C'].loc[training_cols].values
    test_carbon_portions = fitting_df['C'].loc[test_cols].values

    silicon_portions = fitting_df['Si'].loc[combined_df.columns].values
    training_silicon_portions = fitting_df['Si'].loc[training_cols].values
    test_silicon_portions = fitting_df['Si'].loc[test_cols].values

    oxygen_portions = fitting_df['O'].loc[combined_df.columns].values
    training_oxygen_portions = fitting_df['O'].loc[training_cols].values
    test_oxygen_portions = fitting_df['O'].loc[test_cols].values

    result, analyze = activation_layer_multivar(np.array([
        training_carbon_portions, 
        # training_silicon_portions, 
        # training_oxygen_portions
        ]).T, np.array(training_true_carbon))

    predicted_carbon = analyze(np.array([
        carbon_portions, 
        # silicon_portions, 
        # oxygen_portions
        ]).T)
    predicted_training_carbon = analyze(np.array([
        training_carbon_portions, 
        # training_silicon_portions, 
        # training_oxygen_portions
        ]).T)
    predicted_test_carbon = analyze(np.array([
        test_carbon_portions, 
        # test_silicon_portions, 
        # test_oxygen_portions
        ]).T)

    mse_test = np.mean(np.abs(test_true_carbon - predicted_test_carbon))

    worst_accuracy = np.max(np.abs(test_true_carbon - predicted_test_carbon))

    info = {
        'mse': mse_test,
        'worst_accuracy': worst_accuracy,
        'fitting_df': fitting_df,
        'predicted_df': predicted_df,
        'combined_df': _combined_df,
        'train_cols': list(elem_exp_df.index),
        'carbon_portions': carbon_portions,
        'silicon_portions': silicon_portions,
        'oxygen_portions': oxygen_portions,
        'training_carbon_portions': training_carbon_portions,
        'test_carbon_portions': test_carbon_portions,
        'true_carbon': true_carbon,
        'true_silicon': true_silicon,
        'true_oxygen': true_oxygen,
        'training_true_carbon': training_true_carbon,
        'test_true_carbon': test_true_carbon,
        'predicted_carbon': predicted_carbon,
        'predicted_training_carbon': predicted_training_carbon,
        'predicted_test_carbon': predicted_test_carbon,
        'method group': 'Component Analysis',
        'Variables': 'C only'
        
    }
    return info

In [26]:
def CAelemConvFull(training_df, test_df, training_exp_df, test_exp_df, elem_df=elem_Readings, elem_exp_df=elem_sims):
    """
    Perform component analysis on the training and test data.
    """
    training_cols = training_df.columns
    test_cols = test_df.columns
    
    combined_df = pd.concat([training_df, test_df], axis=1)
    combined_df = combined_df.loc[:, ~combined_df.columns.duplicated()]
    combined_exp_df = pd.concat([training_exp_df, test_exp_df], axis=0)
    combined_exp_df = combined_exp_df.loc[~combined_exp_df.index.duplicated(keep='first')]

    true_carbon = combined_exp_df['carbon_level'].values
    test_true_carbon = test_exp_df['carbon_level'].values
    training_true_carbon = training_exp_df['carbon_level'].values
    
    true_silicon = combined_exp_df['silicon_level'].values
    test_true_silicon = test_exp_df['silicon_level'].values
    training_true_silicon = training_exp_df['silicon_level'].values

    true_oxygen = combined_exp_df['oxygen_level'].values
    test_true_oxygen = test_exp_df['oxygen_level'].values
    training_true_oxygen = training_exp_df['oxygen_level'].values

    elem_df = elem_df.loc[combined_df.index]
    _combined_df = pd.concat([combined_df, elem_df], axis=1)
    _combined_exp_df = pd.concat([combined_exp_df, elem_exp_df], axis=0)
    
    fitting_df, predicted_df = ca.Analyze(
        _combined_df,
        bins=combined_df.index,
        train_cols=list(elem_exp_df.index),
        true_c_concentrations=_combined_exp_df['carbon_level'].values,
        normalize=False,
        window = [1.0, 7],
        minimizer='convex_relative'
    )
    
    carbon_portions = fitting_df['C'].loc[combined_df.columns].values
    training_carbon_portions = fitting_df['C'].loc[training_cols].values
    test_carbon_portions = fitting_df['C'].loc[test_cols].values

    silicon_portions = fitting_df['Si'].loc[combined_df.columns].values
    training_silicon_portions = fitting_df['Si'].loc[training_cols].values
    test_silicon_portions = fitting_df['Si'].loc[test_cols].values

    oxygen_portions = fitting_df['O'].loc[combined_df.columns].values
    training_oxygen_portions = fitting_df['O'].loc[training_cols].values
    test_oxygen_portions = fitting_df['O'].loc[test_cols].values

    result, analyze = activation_layer_multivar(np.array([training_carbon_portions, training_silicon_portions, training_oxygen_portions]).T, np.array(training_true_carbon))

    predicted_carbon = analyze(np.array([fitting_df['C'].loc[combined_df.columns].values, fitting_df['Si'].loc[combined_df.columns].values, fitting_df['O'].loc[combined_df.columns].values]).T)
    predicted_training_carbon = analyze(np.array([fitting_df['C'].loc[training_cols].values, fitting_df['Si'].loc[training_cols].values, fitting_df['O'].loc[training_cols].values]).T)
    predicted_test_carbon = analyze(np.array([fitting_df['C'].loc[test_cols].values, fitting_df['Si'].loc[test_cols].values, fitting_df['O'].loc[test_cols].values]).T)

    mse_test = np.mean(np.abs(test_true_carbon - predicted_test_carbon))

    worst_accuracy = np.max(np.abs(test_true_carbon - predicted_test_carbon))

    info = {
        'mse': mse_test,
        'worst_accuracy': worst_accuracy,
        'fitting_df': fitting_df,
        'predicted_df': predicted_df,
        'combined_df': _combined_df,
        'train_cols': list(elem_exp_df.index),
        'carbon_portions': carbon_portions,
        'silicon_portions': silicon_portions,
        'oxygen_portions': oxygen_portions,
        'training_carbon_portions': training_carbon_portions,
        'test_carbon_portions': test_carbon_portions,
        'true_carbon': true_carbon,
        'true_silicon': true_silicon,
        'true_oxygen': true_oxygen,
        'training_true_carbon': training_true_carbon,
        'test_true_carbon': test_true_carbon,
        'predicted_carbon': predicted_carbon,
        'predicted_training_carbon': predicted_training_carbon,
        'predicted_test_carbon': predicted_test_carbon,
        'method group': 'Component Analysis',
        'Variables': 'C Si and O'
        
    }
    return info

# Combined

In [27]:
def PF_Exp_Falloff_CAelem(training_df, test_df, training_exp_df, test_exp_df, elem_df=elem_Readings, elem_exp_df=elem_sims):
    """
    Perform peak fitting on the training and test data.
    """
    training_cols = training_df.columns
    test_cols = test_df.columns
    
    combined_df = pd.concat([training_df, test_df], axis=1)
    combined_df = combined_df.loc[:, ~combined_df.columns.duplicated()]
    combined_exp_df = pd.concat([training_exp_df, test_exp_df], axis=0)
    combined_exp_df = combined_exp_df.loc[~combined_exp_df.index.duplicated(keep='first')]

    true_carbon = combined_exp_df['carbon_level'].values
    test_true_carbon = test_exp_df['carbon_level'].values
    training_true_carbon = training_exp_df['carbon_level'].values
    
    true_silicon = combined_exp_df['silicon_level'].values
    test_true_silicon = test_exp_df['silicon_level'].values
    training_true_silicon = training_exp_df['silicon_level'].values

    true_oxygen = combined_exp_df['oxygen_level'].values
    test_true_oxygen = test_exp_df['oxygen_level'].values
    training_true_oxygen = training_exp_df['oxygen_level'].values

    elem_df = elem_df.loc[combined_df.index]
    _combined_df = pd.concat([combined_df, elem_df], axis=1)
    _combined_exp_df = pd.concat([combined_exp_df, elem_exp_df], axis=0)
    
    fitting_df, predicted_df = ca.Analyze(
        _combined_df,
        bins=combined_df.index,
        train_cols=list(elem_exp_df.index),
        true_c_concentrations=_combined_exp_df['carbon_level'].values,
        normalize=False,
        window = [1.0, 7]
    )

    carbon_portions = fitting_df['C'].loc[combined_df.columns].values
    training_carbon_portions = fitting_df['C'].loc[training_cols].values
    test_carbon_portions = fitting_df['C'].loc[test_cols].values

    silicon_portions = fitting_df['Si'].loc[combined_df.columns].values
    training_silicon_portions = fitting_df['Si'].loc[training_cols].values
    test_silicon_portions = fitting_df['Si'].loc[test_cols].values

    oxygen_portions = fitting_df['O'].loc[combined_df.columns].values
    training_oxygen_portions = fitting_df['O'].loc[training_cols].values
    test_oxygen_portions = fitting_df['O'].loc[test_cols].values

    peak_fitting_df1, carbon_fitting_df, si_fitting_df, c_lines_df, si_lines_df = pfa.PeakFit(
        combined_df,
        bins=combined_df.index,
        c_window=[4.2, 4.7], 
        si_window=[1.5, 2],
        c_baseline='exp_falloff',
        si_baseline='exp_falloff',
        )
    
    peak_fitting_df2, o_fitting_df, _fitting_df, o_lines_df, _lines_df = pfa.PeakFit(
        combined_df,
        bins=combined_df.index,
        c_window=[5.9, 6.3], 
        si_window=[1.5, 2],
        c_baseline='exp_falloff',
        si_baseline='exp_falloff',
        )
    

    peak_fitting_df1 = peak_fitting_df1.set_index('label')
    training_peak_fitting_df = peak_fitting_df1.loc[training_cols]
    test_peak_fitting_df = peak_fitting_df1.loc[test_cols]

    carbon_peak_areas = peak_fitting_df1['Carbon Peak Area'].values
    training_carbon_peak_areas = training_peak_fitting_df['Carbon Peak Area'].values
    test_carbon_peak_areas = test_peak_fitting_df['Carbon Peak Area'].values

    silicon_peak_areas = peak_fitting_df1['Silicon Peak Area'].values
    training_silicon_peak_areas = training_peak_fitting_df['Silicon Peak Area'].values
    test_silicon_peak_areas = test_peak_fitting_df['Silicon Peak Area'].values

    peak_fitting_df2 = peak_fitting_df2.set_index('label')
    training_peak_fitting_df2 = peak_fitting_df2.loc[training_cols]
    test_peak_fitting_df2 = peak_fitting_df2.loc[test_cols]

    oxygen_peak_areas = peak_fitting_df2['Carbon Peak Area'].values
    training_oxygen_peak_areas = training_peak_fitting_df2['Carbon Peak Area'].values
    test_oxygen_peak_areas = test_peak_fitting_df2['Carbon Peak Area'].values


    result, analyze = activation_layer_multivar(np.array([
        training_carbon_portions, training_silicon_portions, training_oxygen_portions,
        training_carbon_peak_areas, training_silicon_peak_areas, training_oxygen_peak_areas
        ]).T, np.array(training_true_carbon))

    predicted_carbon = analyze(np.array([
        carbon_portions, silicon_portions, oxygen_portions,
        carbon_peak_areas, silicon_peak_areas, oxygen_peak_areas
        ]).T)
    predicted_training_carbon = analyze(np.array([
        training_carbon_portions, training_silicon_portions, training_oxygen_portions,
        training_carbon_peak_areas, training_silicon_peak_areas, training_oxygen_peak_areas
        ]).T)
    predicted_test_carbon = analyze(np.array([
        test_carbon_portions, test_silicon_portions, test_oxygen_portions,
        test_carbon_peak_areas, test_silicon_peak_areas, test_oxygen_peak_areas
        ]).T)

    mse_test = np.mean(np.abs(test_true_carbon - predicted_test_carbon))

    worst_accuracy = np.max(np.abs(test_true_carbon - predicted_test_carbon))

    info = {
        'mse': mse_test,
        'worst_accuracy': worst_accuracy,
        'fitting_df': fitting_df,
        'predicted_df': predicted_df,
        'combined_df': _combined_df,
        'train_cols': list(elem_exp_df.index),
        'carbon_portions': carbon_portions,
        'silicon_portions': silicon_portions,
        'oxygen_portions': oxygen_portions,
        'carbon_peak_areas': carbon_peak_areas,
        'silicon_peak_areas': silicon_peak_areas,
        'oxygen_peak_areas': oxygen_peak_areas,
        'training_carbon_portions': training_carbon_portions,
        'test_carbon_portions': test_carbon_portions,
        'true_carbon': true_carbon,
        'true_silicon': true_silicon,
        'true_oxygen': true_oxygen,
        'training_true_carbon': training_true_carbon,
        'test_true_carbon': test_true_carbon,
        'predicted_carbon': predicted_carbon,
        'predicted_training_carbon': predicted_training_carbon,
        'predicted_test_carbon': predicted_test_carbon,
        'method group': 'Compound Method',
        
    }
    return info

In [28]:
def FullFeatures(training_df, test_df, training_exp_df, test_exp_df, elem_df=elem_Readings, elem_exp_df=elem_sims):
    """
    Perform peak fitting on the training and test data.
    """
    training_cols = training_df.columns
    test_cols = test_df.columns
    
    combined_df = pd.concat([training_df, test_df], axis=1)
    combined_df = combined_df.loc[:, ~combined_df.columns.duplicated()]
    combined_exp_df = pd.concat([training_exp_df, test_exp_df], axis=0)
    combined_exp_df = combined_exp_df.loc[~combined_exp_df.index.duplicated(keep='first')]

    true_carbon = combined_exp_df['carbon_level'].values
    test_true_carbon = test_exp_df['carbon_level'].values
    training_true_carbon = training_exp_df['carbon_level'].values
    
    true_silicon = combined_exp_df['silicon_level'].values
    test_true_silicon = test_exp_df['silicon_level'].values
    training_true_silicon = training_exp_df['silicon_level'].values

    true_oxygen = combined_exp_df['oxygen_level'].values
    test_true_oxygen = test_exp_df['oxygen_level'].values
    training_true_oxygen = training_exp_df['oxygen_level'].values

    elem_df = elem_df.loc[combined_df.index]
    _combined_df = pd.concat([combined_df, elem_df], axis=1)
    _combined_exp_df = pd.concat([combined_exp_df, elem_exp_df], axis=0)
    
    fitting_df, predicted_df = ca.Analyze(
        _combined_df,
        bins=combined_df.index,
        train_cols=list(elem_exp_df.index),
        true_c_concentrations=_combined_exp_df['carbon_level'].values,
        normalize=False,
        window = [1.0, 7],
        # minimizer='convex_relative'
    )

    carbon_portions = fitting_df['C'].loc[combined_df.columns].values
    training_carbon_portions = fitting_df['C'].loc[training_cols].values
    test_carbon_portions = fitting_df['C'].loc[test_cols].values

    silicon_portions = fitting_df['Si'].loc[combined_df.columns].values
    training_silicon_portions = fitting_df['Si'].loc[training_cols].values
    test_silicon_portions = fitting_df['Si'].loc[test_cols].values

    oxygen_portions = fitting_df['O'].loc[combined_df.columns].values
    training_oxygen_portions = fitting_df['O'].loc[training_cols].values
    test_oxygen_portions = fitting_df['O'].loc[test_cols].values

    peak_fitting_df1, carbon_fitting_df, si_fitting_df, c_lines_df, si_lines_df = pfa.PeakFit(
        combined_df,
        bins=combined_df.index,
        c_window=[4.2, 4.7], 
        si_window=[1.5, 2],
        c_baseline='exp_falloff',
        si_baseline='exp_falloff',
        )
    
    peak_fitting_df2, o_fitting_df, _fitting_df, o_lines_df, _lines_df = pfa.PeakFit(
        combined_df,
        bins=combined_df.index,
        c_window=[5.9, 6.3], 
        si_window=[1.5, 2],
        c_baseline='exp_falloff',
        si_baseline='exp_falloff',
        )
    

    peak_fitting_df1 = peak_fitting_df1.set_index('label')
    training_peak_fitting_df = peak_fitting_df1.loc[training_cols]
    test_peak_fitting_df = peak_fitting_df1.loc[test_cols]

    carbon_peak_areas = peak_fitting_df1['Carbon Peak Area'].values
    training_carbon_peak_areas = training_peak_fitting_df['Carbon Peak Area'].values
    test_carbon_peak_areas = test_peak_fitting_df['Carbon Peak Area'].values

    silicon_peak_areas = peak_fitting_df1['Silicon Peak Area'].values
    training_silicon_peak_areas = training_peak_fitting_df['Silicon Peak Area'].values
    test_silicon_peak_areas = test_peak_fitting_df['Silicon Peak Area'].values

    peak_fitting_df2 = peak_fitting_df2.set_index('label')
    training_peak_fitting_df2 = peak_fitting_df2.loc[training_cols]
    test_peak_fitting_df2 = peak_fitting_df2.loc[test_cols]

    oxygen_peak_areas = peak_fitting_df2['Carbon Peak Area'].values
    training_oxygen_peak_areas = training_peak_fitting_df2['Carbon Peak Area'].values
    test_oxygen_peak_areas = test_peak_fitting_df2['Carbon Peak Area'].values


    result, analyze = activation_layer_multivar(np.array([
        training_carbon_portions, training_silicon_portions, training_oxygen_portions,
        training_carbon_peak_areas, training_silicon_peak_areas, training_oxygen_peak_areas
        ]).T, np.array(training_true_carbon))

    predicted_carbon = analyze(np.array([
        carbon_portions, silicon_portions, oxygen_portions,
        carbon_peak_areas, silicon_peak_areas, oxygen_peak_areas
        ]).T)
    predicted_training_carbon = analyze(np.array([
        training_carbon_portions, training_silicon_portions, training_oxygen_portions,
        training_carbon_peak_areas, training_silicon_peak_areas, training_oxygen_peak_areas
        ]).T)
    predicted_test_carbon = analyze(np.array([
        test_carbon_portions, test_silicon_portions, test_oxygen_portions,
        test_carbon_peak_areas, test_silicon_peak_areas, test_oxygen_peak_areas
        ]).T)

    mse_test = np.mean(np.abs(test_true_carbon - predicted_test_carbon))

    worst_accuracy = np.max(np.abs(test_true_carbon - predicted_test_carbon))

    info = {
        'mse': mse_test,
        'worst_accuracy': worst_accuracy,
        'fitting_df': fitting_df,
        'predicted_df': predicted_df,
        'combined_df': _combined_df,
        'train_cols': list(elem_exp_df.index),
        'carbon_portions': carbon_portions,
        'silicon_portions': silicon_portions,
        'oxygen_portions': oxygen_portions,
        'carbon_peak_areas': carbon_peak_areas,
        'silicon_peak_areas': silicon_peak_areas,
        'oxygen_peak_areas': oxygen_peak_areas,
        'training_carbon_portions': training_carbon_portions,
        'test_carbon_portions': test_carbon_portions,
        'true_carbon': true_carbon,
        'true_silicon': true_silicon,
        'true_oxygen': true_oxygen,
        'training_true_carbon': training_true_carbon,
        'test_true_carbon': test_true_carbon,
        'predicted_carbon': predicted_carbon,
        'predicted_training_carbon': predicted_training_carbon,
        'predicted_test_carbon': predicted_test_carbon,
        'method group': 'Compound Method',
        
    }
    return info

In [29]:
def FullFeaturesConvex(training_df, test_df, training_exp_df, test_exp_df, elem_df=elem_Readings, elem_exp_df=elem_sims):
    """
    Perform peak fitting on the training and test data.
    """
    training_cols = training_df.columns
    test_cols = test_df.columns
    
    combined_df = pd.concat([training_df, test_df], axis=1)
    combined_df = combined_df.loc[:, ~combined_df.columns.duplicated()]
    combined_exp_df = pd.concat([training_exp_df, test_exp_df], axis=0)
    combined_exp_df = combined_exp_df.loc[~combined_exp_df.index.duplicated(keep='first')]

    true_carbon = combined_exp_df['carbon_level'].values
    test_true_carbon = test_exp_df['carbon_level'].values
    training_true_carbon = training_exp_df['carbon_level'].values
    
    true_silicon = combined_exp_df['silicon_level'].values
    test_true_silicon = test_exp_df['silicon_level'].values
    training_true_silicon = training_exp_df['silicon_level'].values

    true_oxygen = combined_exp_df['oxygen_level'].values
    test_true_oxygen = test_exp_df['oxygen_level'].values
    training_true_oxygen = training_exp_df['oxygen_level'].values

    elem_df = elem_df.loc[combined_df.index]
    _combined_df = pd.concat([combined_df, elem_df], axis=1)
    _combined_exp_df = pd.concat([combined_exp_df, elem_exp_df], axis=0)
    
    fitting_df, predicted_df = ca.Analyze(
        _combined_df,
        bins=combined_df.index,
        train_cols=list(elem_exp_df.index),
        true_c_concentrations=_combined_exp_df['carbon_level'].values,
        normalize=False,
        window = [1.0, 7],
        minimizer='convex_relative'
    )

    carbon_portions = fitting_df['C'].loc[combined_df.columns].values
    training_carbon_portions = fitting_df['C'].loc[training_cols].values
    test_carbon_portions = fitting_df['C'].loc[test_cols].values

    silicon_portions = fitting_df['Si'].loc[combined_df.columns].values
    training_silicon_portions = fitting_df['Si'].loc[training_cols].values
    test_silicon_portions = fitting_df['Si'].loc[test_cols].values

    oxygen_portions = fitting_df['O'].loc[combined_df.columns].values
    training_oxygen_portions = fitting_df['O'].loc[training_cols].values
    test_oxygen_portions = fitting_df['O'].loc[test_cols].values

    peak_fitting_df1, carbon_fitting_df, si_fitting_df, c_lines_df, si_lines_df = pfa.PeakFit(
        combined_df,
        bins=combined_df.index,
        c_window=[4.2, 4.7], 
        si_window=[1.5, 2],
        c_baseline='exp_falloff',
        si_baseline='exp_falloff',
        )
    
    peak_fitting_df2, o_fitting_df, _fitting_df, o_lines_df, _lines_df = pfa.PeakFit(
        combined_df,
        bins=combined_df.index,
        c_window=[5.9, 6.3], 
        si_window=[1.5, 2],
        c_baseline='exp_falloff',
        si_baseline='exp_falloff',
        )
    

    peak_fitting_df1 = peak_fitting_df1.set_index('label')
    training_peak_fitting_df = peak_fitting_df1.loc[training_cols]
    test_peak_fitting_df = peak_fitting_df1.loc[test_cols]

    carbon_peak_areas = peak_fitting_df1['Carbon Peak Area'].values
    training_carbon_peak_areas = training_peak_fitting_df['Carbon Peak Area'].values
    test_carbon_peak_areas = test_peak_fitting_df['Carbon Peak Area'].values

    silicon_peak_areas = peak_fitting_df1['Silicon Peak Area'].values
    training_silicon_peak_areas = training_peak_fitting_df['Silicon Peak Area'].values
    test_silicon_peak_areas = test_peak_fitting_df['Silicon Peak Area'].values

    peak_fitting_df2 = peak_fitting_df2.set_index('label')
    training_peak_fitting_df2 = peak_fitting_df2.loc[training_cols]
    test_peak_fitting_df2 = peak_fitting_df2.loc[test_cols]

    oxygen_peak_areas = peak_fitting_df2['Carbon Peak Area'].values
    training_oxygen_peak_areas = training_peak_fitting_df2['Carbon Peak Area'].values
    test_oxygen_peak_areas = test_peak_fitting_df2['Carbon Peak Area'].values


    result, analyze = activation_layer_multivar(np.array([
        training_carbon_portions, training_silicon_portions, training_oxygen_portions,
        training_carbon_peak_areas, training_silicon_peak_areas, training_oxygen_peak_areas
        ]).T, np.array(training_true_carbon))

    predicted_carbon = analyze(np.array([
        carbon_portions, silicon_portions, oxygen_portions,
        carbon_peak_areas, silicon_peak_areas, oxygen_peak_areas
        ]).T)
    predicted_training_carbon = analyze(np.array([
        training_carbon_portions, training_silicon_portions, training_oxygen_portions,
        training_carbon_peak_areas, training_silicon_peak_areas, training_oxygen_peak_areas
        ]).T)
    predicted_test_carbon = analyze(np.array([
        test_carbon_portions, test_silicon_portions, test_oxygen_portions,
        test_carbon_peak_areas, test_silicon_peak_areas, test_oxygen_peak_areas
        ]).T)

    mse_test = np.mean(np.abs(test_true_carbon - predicted_test_carbon))

    worst_accuracy = np.max(np.abs(test_true_carbon - predicted_test_carbon))

    info = {
        'mse': mse_test,
        'worst_accuracy': worst_accuracy,
        'fitting_df': fitting_df,
        'predicted_df': predicted_df,
        'combined_df': _combined_df,
        'train_cols': list(elem_exp_df.index),
        'carbon_portions': carbon_portions,
        'silicon_portions': silicon_portions,
        'oxygen_portions': oxygen_portions,
        'carbon_peak_areas': carbon_peak_areas,
        'silicon_peak_areas': silicon_peak_areas,
        'oxygen_peak_areas': oxygen_peak_areas,
        'training_carbon_portions': training_carbon_portions,
        'test_carbon_portions': test_carbon_portions,
        'true_carbon': true_carbon,
        'true_silicon': true_silicon,
        'true_oxygen': true_oxygen,
        'training_true_carbon': training_true_carbon,
        'test_true_carbon': test_true_carbon,
        'predicted_carbon': predicted_carbon,
        'predicted_training_carbon': predicted_training_carbon,
        'predicted_test_carbon': predicted_test_carbon,
        'method group': 'Compound Method',
        
    }
    return info

In [30]:
def PF_Exp_Falloff_CAelem_regression(training_df, test_df, training_exp_df, test_exp_df, elem_df=elem_Readings, elem_exp_df=elem_sims):
    """
    Perform peak fitting on the training and test data.
    """
    training_cols = training_df.columns
    test_cols = test_df.columns
    
    combined_df = pd.concat([training_df, test_df], axis=1)
    combined_df = combined_df.loc[:, ~combined_df.columns.duplicated()]
    combined_exp_df = pd.concat([training_exp_df, test_exp_df], axis=0)
    combined_exp_df = combined_exp_df.loc[~combined_exp_df.index.duplicated(keep='first')]

    true_carbon = combined_exp_df['carbon_level'].values
    test_true_carbon = test_exp_df['carbon_level'].values
    training_true_carbon = training_exp_df['carbon_level'].values
    
    true_silicon = combined_exp_df['silicon_level'].values
    test_true_silicon = test_exp_df['silicon_level'].values
    training_true_silicon = training_exp_df['silicon_level'].values

    true_oxygen = combined_exp_df['oxygen_level'].values
    test_true_oxygen = test_exp_df['oxygen_level'].values
    training_true_oxygen = training_exp_df['oxygen_level'].values

    elem_df = elem_df.loc[combined_df.index]
    _combined_df = pd.concat([combined_df, elem_df], axis=1)
    _combined_exp_df = pd.concat([combined_exp_df, elem_exp_df], axis=0)
    
    fitting_df, predicted_df = ca.Analyze(
        _combined_df,
        bins=combined_df.index,
        train_cols=list(elem_exp_df.index),
        true_c_concentrations=_combined_exp_df['carbon_level'].values,
        normalize=False,
        window = [1.0, 7],
        # minimizer='convex_relative'
    )

    carbon_portions = fitting_df['C'].loc[combined_df.columns].values
    training_carbon_portions = fitting_df['C'].loc[training_cols].values
    test_carbon_portions = fitting_df['C'].loc[test_cols].values

    silicon_portions = fitting_df['Si'].loc[combined_df.columns].values
    training_silicon_portions = fitting_df['Si'].loc[training_cols].values
    test_silicon_portions = fitting_df['Si'].loc[test_cols].values

    oxygen_portions = fitting_df['O'].loc[combined_df.columns].values
    training_oxygen_portions = fitting_df['O'].loc[training_cols].values
    test_oxygen_portions = fitting_df['O'].loc[test_cols].values

    peak_fitting_df1, carbon_fitting_df, si_fitting_df, c_lines_df, si_lines_df = pfa.PeakFit(
        combined_df,
        bins=combined_df.index,
        c_window=[4.2, 4.7], 
        si_window=[1.5, 2],
        c_baseline='exp_falloff',
        si_baseline='exp_falloff',
        )
    
    peak_fitting_df2, o_fitting_df, _fitting_df, o_lines_df, _lines_df = pfa.PeakFit(
        combined_df,
        bins=combined_df.index,
        c_window=[5.9, 6.3], 
        si_window=[1.5, 2],
        c_baseline='exp_falloff',
        si_baseline='exp_falloff',
        )
    

    peak_fitting_df1 = peak_fitting_df1.set_index('label')
    training_peak_fitting_df = peak_fitting_df1.loc[training_cols]
    test_peak_fitting_df = peak_fitting_df1.loc[test_cols]

    carbon_peak_areas = peak_fitting_df1['Carbon Peak Area'].values
    training_carbon_peak_areas = training_peak_fitting_df['Carbon Peak Area'].values
    test_carbon_peak_areas = test_peak_fitting_df['Carbon Peak Area'].values

    silicon_peak_areas = peak_fitting_df1['Silicon Peak Area'].values
    training_silicon_peak_areas = training_peak_fitting_df['Silicon Peak Area'].values
    test_silicon_peak_areas = test_peak_fitting_df['Silicon Peak Area'].values

    peak_fitting_df2 = peak_fitting_df2.set_index('label')
    training_peak_fitting_df2 = peak_fitting_df2.loc[training_cols]
    test_peak_fitting_df2 = peak_fitting_df2.loc[test_cols]

    oxygen_peak_areas = peak_fitting_df2['Carbon Peak Area'].values
    training_oxygen_peak_areas = training_peak_fitting_df2['Carbon Peak Area'].values
    test_oxygen_peak_areas = test_peak_fitting_df2['Carbon Peak Area'].values


    result, analyze = activation_layer_multivar_linreg(np.array([
        training_carbon_portions, training_silicon_portions, training_oxygen_portions,
        training_carbon_peak_areas, training_silicon_peak_areas, training_oxygen_peak_areas
        ]).T, np.array(training_true_carbon))

    predicted_carbon = analyze(np.array([
        carbon_portions, silicon_portions, oxygen_portions,
        carbon_peak_areas, silicon_peak_areas, oxygen_peak_areas
        ]).T)
    predicted_training_carbon = analyze(np.array([
        training_carbon_portions, training_silicon_portions, training_oxygen_portions,
        training_carbon_peak_areas, training_silicon_peak_areas, training_oxygen_peak_areas
        ]).T)
    predicted_test_carbon = analyze(np.array([
        test_carbon_portions, test_silicon_portions, test_oxygen_portions,
        test_carbon_peak_areas, test_silicon_peak_areas, test_oxygen_peak_areas
        ]).T)

    mse_test = np.mean(np.abs(test_true_carbon - predicted_test_carbon))

    worst_accuracy = np.max(np.abs(test_true_carbon - predicted_test_carbon))

    info = {
        'mse': mse_test,
        'worst_accuracy': worst_accuracy,
        'fitting_df': fitting_df,
        'predicted_df': predicted_df,
        'combined_df': _combined_df,
        'train_cols': list(elem_exp_df.index),
        'carbon_portions': carbon_portions,
        'silicon_portions': silicon_portions,
        'oxygen_portions': oxygen_portions,
        'carbon_peak_areas': carbon_peak_areas,
        'silicon_peak_areas': silicon_peak_areas,
        'oxygen_peak_areas': oxygen_peak_areas,
        'training_carbon_portions': training_carbon_portions,
        'test_carbon_portions': test_carbon_portions,
        'true_carbon': true_carbon,
        'true_silicon': true_silicon,
        'true_oxygen': true_oxygen,
        'training_true_carbon': training_true_carbon,
        'test_true_carbon': test_true_carbon,
        'predicted_carbon': predicted_carbon,
        'predicted_training_carbon': predicted_training_carbon,
        'predicted_test_carbon': predicted_test_carbon,
        'method group': 'Compound Method',
        
    }
    return info

In [31]:
def PF_Exp_Falloff_CAelem_featurescaling(training_df, test_df, training_exp_df, test_exp_df, elem_df=elem_Readings, elem_exp_df=elem_sims):
    """
    Perform peak fitting on the training and test data.
    """
    training_cols = training_df.columns
    test_cols = test_df.columns
    
    combined_df = pd.concat([training_df, test_df], axis=1)
    combined_df = combined_df.loc[:, ~combined_df.columns.duplicated()]
    combined_exp_df = pd.concat([training_exp_df, test_exp_df], axis=0)
    combined_exp_df = combined_exp_df.loc[~combined_exp_df.index.duplicated(keep='first')]

    true_carbon = combined_exp_df['carbon_level'].values
    test_true_carbon = test_exp_df['carbon_level'].values
    training_true_carbon = training_exp_df['carbon_level'].values
    
    true_silicon = combined_exp_df['silicon_level'].values
    test_true_silicon = test_exp_df['silicon_level'].values
    training_true_silicon = training_exp_df['silicon_level'].values

    true_oxygen = combined_exp_df['oxygen_level'].values
    test_true_oxygen = test_exp_df['oxygen_level'].values
    training_true_oxygen = training_exp_df['oxygen_level'].values

    elem_df = elem_df.loc[combined_df.index]
    _combined_df = pd.concat([combined_df, elem_df], axis=1)
    _combined_exp_df = pd.concat([combined_exp_df, elem_exp_df], axis=0)
    
    fitting_df, predicted_df = ca.Analyze(
        _combined_df,
        bins=combined_df.index,
        train_cols=list(elem_exp_df.index),
        true_c_concentrations=_combined_exp_df['carbon_level'].values,
        normalize=False,
        window = [1.0, 7],
        # minimizer='convex_relative'
    )

    carbon_portions = fitting_df['C'].loc[combined_df.columns].values
    training_carbon_portions = fitting_df['C'].loc[training_cols].values
    test_carbon_portions = fitting_df['C'].loc[test_cols].values

    silicon_portions = fitting_df['Si'].loc[combined_df.columns].values
    training_silicon_portions = fitting_df['Si'].loc[training_cols].values
    test_silicon_portions = fitting_df['Si'].loc[test_cols].values

    oxygen_portions = fitting_df['O'].loc[combined_df.columns].values
    training_oxygen_portions = fitting_df['O'].loc[training_cols].values
    test_oxygen_portions = fitting_df['O'].loc[test_cols].values

    peak_fitting_df1, carbon_fitting_df, si_fitting_df, c_lines_df, si_lines_df = pfa.PeakFit(
        combined_df,
        bins=combined_df.index,
        c_window=[4.2, 4.7], 
        si_window=[1.5, 2],
        c_baseline='exp_falloff',
        si_baseline='exp_falloff',
        )
    
    peak_fitting_df2, o_fitting_df, _fitting_df, o_lines_df, _lines_df = pfa.PeakFit(
        combined_df,
        bins=combined_df.index,
        c_window=[5.9, 6.3], 
        si_window=[1.5, 2],
        c_baseline='exp_falloff',
        si_baseline='exp_falloff',
        )
    

    peak_fitting_df1 = peak_fitting_df1.set_index('label')
    training_peak_fitting_df = peak_fitting_df1.loc[training_cols]
    test_peak_fitting_df = peak_fitting_df1.loc[test_cols]

    carbon_peak_areas = peak_fitting_df1['Carbon Peak Area'].values
    training_carbon_peak_areas = training_peak_fitting_df['Carbon Peak Area'].values
    test_carbon_peak_areas = test_peak_fitting_df['Carbon Peak Area'].values

    silicon_peak_areas = peak_fitting_df1['Silicon Peak Area'].values
    training_silicon_peak_areas = training_peak_fitting_df['Silicon Peak Area'].values
    test_silicon_peak_areas = test_peak_fitting_df['Silicon Peak Area'].values

    peak_fitting_df2 = peak_fitting_df2.set_index('label')
    training_peak_fitting_df2 = peak_fitting_df2.loc[training_cols]
    test_peak_fitting_df2 = peak_fitting_df2.loc[test_cols]

    oxygen_peak_areas = peak_fitting_df2['Carbon Peak Area'].values
    training_oxygen_peak_areas = training_peak_fitting_df2['Carbon Peak Area'].values
    test_oxygen_peak_areas = test_peak_fitting_df2['Carbon Peak Area'].values

    # Feature scaling
    _training_carbon_portions = (training_carbon_portions-np.min(training_carbon_portions)) / (np.max(training_carbon_portions)-np.min(training_carbon_portions))
    _carbon_portions = (carbon_portions-np.min(training_carbon_portions)) / (np.max(training_carbon_portions)-np.min(training_carbon_portions))
    _test_carbon_portions = (test_carbon_portions-np.min(training_carbon_portions)) / (np.max(training_carbon_portions)-np.min(training_carbon_portions))

    _training_silicon_portions = (training_silicon_portions-np.min(training_silicon_portions)) / (np.max(training_silicon_portions)-np.min(training_silicon_portions))
    _silicon_portions = (silicon_portions-np.min(training_silicon_portions)) / (np.max(training_silicon_portions)-np.min(training_silicon_portions))
    _test_silicon_portions = (test_silicon_portions-np.min(training_silicon_portions)) / (np.max(training_silicon_portions)-np.min(training_silicon_portions))

    _training_oxygen_portions = (training_oxygen_portions-np.min(training_oxygen_portions)) / (np.max(training_oxygen_portions)-np.min(training_oxygen_portions))
    _oxygen_portions = (oxygen_portions-np.min(training_oxygen_portions)) / (np.max(training_oxygen_portions)-np.min(training_oxygen_portions))
    _test_oxygen_portions = (test_oxygen_portions-np.min(training_oxygen_portions)) / (np.max(training_oxygen_portions)-np.min(training_oxygen_portions))
    
    _training_carbon_peak_areas = (training_carbon_peak_areas-np.min(training_carbon_peak_areas)) / (np.max(training_carbon_peak_areas)-np.min(training_carbon_peak_areas))
    _carbon_peak_areas = (carbon_peak_areas-np.min(training_carbon_peak_areas)) / (np.max(training_carbon_peak_areas)-np.min(training_carbon_peak_areas))
    _test_carbon_peak_areas = (test_carbon_peak_areas-np.min(training_carbon_peak_areas)) / (np.max(training_carbon_peak_areas)-np.min(training_carbon_peak_areas))
    _training_silicon_peak_areas = (training_silicon_peak_areas-np.min(training_silicon_peak_areas)) / (np.max(training_silicon_peak_areas)-np.min(training_silicon_peak_areas))
    _silicon_peak_areas = (silicon_peak_areas-np.min(training_silicon_peak_areas)) / (np.max(training_silicon_peak_areas)-np.min(training_silicon_peak_areas))
    _test_silicon_peak_areas = (test_silicon_peak_areas-np.min(training_silicon_peak_areas)) / (np.max(training_silicon_peak_areas)-np.min(training_silicon_peak_areas))
    _training_oxygen_peak_areas = (training_oxygen_peak_areas-np.min(training_oxygen_peak_areas)) / (np.max(training_oxygen_peak_areas)-np.min(training_oxygen_peak_areas))
    _oxygen_peak_areas = (oxygen_peak_areas-np.min(training_oxygen_peak_areas)) / (np.max(training_oxygen_peak_areas)-np.min(training_oxygen_peak_areas))
    _test_oxygen_peak_areas = (test_oxygen_peak_areas-np.min(training_oxygen_peak_areas)) / (np.max(training_oxygen_peak_areas)-np.min(training_oxygen_peak_areas))

    result, analyze = activation_layer_multivar(np.array([
        _training_carbon_portions, _training_silicon_portions, _training_oxygen_portions,
        _training_carbon_peak_areas, _training_silicon_peak_areas, _training_oxygen_peak_areas
        ]).T, np.array(training_true_carbon))

    predicted_carbon = analyze(np.array([
        _carbon_portions, _silicon_portions, _oxygen_portions,
        _carbon_peak_areas, _silicon_peak_areas, _oxygen_peak_areas
        ]).T)
    predicted_training_carbon = analyze(np.array([
        _training_carbon_portions, _training_silicon_portions, _training_oxygen_portions,
        _training_carbon_peak_areas, _training_silicon_peak_areas, _training_oxygen_peak_areas
        ]).T)
    predicted_test_carbon = analyze(np.array([
        _test_carbon_portions, _test_silicon_portions, _test_oxygen_portions,
        _test_carbon_peak_areas, _test_silicon_peak_areas, _test_oxygen_peak_areas
        ]).T)

    mse_test = np.mean(np.abs(test_true_carbon - predicted_test_carbon))

    worst_accuracy = np.max(np.abs(test_true_carbon - predicted_test_carbon))

    info = {
        'mse': mse_test,
        'worst_accuracy': worst_accuracy,
        'fitting_df': fitting_df,
        'predicted_df': predicted_df,
        'combined_df': _combined_df,
        'train_cols': list(elem_exp_df.index),
        'carbon_portions': carbon_portions,
        'silicon_portions': silicon_portions,
        'oxygen_portions': oxygen_portions,
        'carbon_peak_areas': carbon_peak_areas,
        'silicon_peak_areas': silicon_peak_areas,
        'oxygen_peak_areas': oxygen_peak_areas,
        'training_carbon_portions': training_carbon_portions,
        'test_carbon_portions': test_carbon_portions,
        'true_carbon': true_carbon,
        'true_silicon': true_silicon,
        'true_oxygen': true_oxygen,
        'training_true_carbon': training_true_carbon,
        'test_true_carbon': test_true_carbon,
        'predicted_carbon': predicted_carbon,
        'predicted_training_carbon': predicted_training_carbon,
        'predicted_test_carbon': predicted_test_carbon,
        'method group': 'Compound Method',
        
    }
    return info

In [32]:
def PF_Exp_Falloff_CAelem_regression_feature_scaling(training_df, test_df, training_exp_df, test_exp_df, elem_df=elem_Readings, elem_exp_df=elem_sims):
    """
    Perform peak fitting on the training and test data.
    """
    training_cols = training_df.columns
    test_cols = test_df.columns
    
    combined_df = pd.concat([training_df, test_df], axis=1)
    combined_df = combined_df.loc[:, ~combined_df.columns.duplicated()]
    combined_exp_df = pd.concat([training_exp_df, test_exp_df], axis=0)
    combined_exp_df = combined_exp_df.loc[~combined_exp_df.index.duplicated(keep='first')]

    true_carbon = combined_exp_df['carbon_level'].values
    test_true_carbon = test_exp_df['carbon_level'].values
    training_true_carbon = training_exp_df['carbon_level'].values
    
    true_silicon = combined_exp_df['silicon_level'].values
    test_true_silicon = test_exp_df['silicon_level'].values
    training_true_silicon = training_exp_df['silicon_level'].values

    true_oxygen = combined_exp_df['oxygen_level'].values
    test_true_oxygen = test_exp_df['oxygen_level'].values
    training_true_oxygen = training_exp_df['oxygen_level'].values

    elem_df = elem_df.loc[combined_df.index]
    _combined_df = pd.concat([combined_df, elem_df], axis=1)
    _combined_exp_df = pd.concat([combined_exp_df, elem_exp_df], axis=0)
    
    fitting_df, predicted_df = ca.Analyze(
        _combined_df,
        bins=combined_df.index,
        train_cols=list(elem_exp_df.index),
        true_c_concentrations=_combined_exp_df['carbon_level'].values,
        normalize=False,
        window = [1.0, 7],
        # minimizer='convex_relative'
    )

    carbon_portions = fitting_df['C'].loc[combined_df.columns].values
    training_carbon_portions = fitting_df['C'].loc[training_cols].values
    test_carbon_portions = fitting_df['C'].loc[test_cols].values

    silicon_portions = fitting_df['Si'].loc[combined_df.columns].values
    training_silicon_portions = fitting_df['Si'].loc[training_cols].values
    test_silicon_portions = fitting_df['Si'].loc[test_cols].values

    oxygen_portions = fitting_df['O'].loc[combined_df.columns].values
    training_oxygen_portions = fitting_df['O'].loc[training_cols].values
    test_oxygen_portions = fitting_df['O'].loc[test_cols].values

    peak_fitting_df1, carbon_fitting_df, si_fitting_df, c_lines_df, si_lines_df = pfa.PeakFit(
        combined_df,
        bins=combined_df.index,
        c_window=[4.2, 4.7], 
        si_window=[1.5, 2],
        c_baseline='exp_falloff',
        si_baseline='exp_falloff',
        )
    
    peak_fitting_df2, o_fitting_df, _fitting_df, o_lines_df, _lines_df = pfa.PeakFit(
        combined_df,
        bins=combined_df.index,
        c_window=[5.9, 6.3], 
        si_window=[1.5, 2],
        c_baseline='exp_falloff',
        si_baseline='exp_falloff',
        )
    

    peak_fitting_df1 = peak_fitting_df1.set_index('label')
    training_peak_fitting_df = peak_fitting_df1.loc[training_cols]
    test_peak_fitting_df = peak_fitting_df1.loc[test_cols]

    carbon_peak_areas = peak_fitting_df1['Carbon Peak Area'].values
    training_carbon_peak_areas = training_peak_fitting_df['Carbon Peak Area'].values
    test_carbon_peak_areas = test_peak_fitting_df['Carbon Peak Area'].values

    silicon_peak_areas = peak_fitting_df1['Silicon Peak Area'].values
    training_silicon_peak_areas = training_peak_fitting_df['Silicon Peak Area'].values
    test_silicon_peak_areas = test_peak_fitting_df['Silicon Peak Area'].values

    peak_fitting_df2 = peak_fitting_df2.set_index('label')
    training_peak_fitting_df2 = peak_fitting_df2.loc[training_cols]
    test_peak_fitting_df2 = peak_fitting_df2.loc[test_cols]

    oxygen_peak_areas = peak_fitting_df2['Carbon Peak Area'].values
    training_oxygen_peak_areas = training_peak_fitting_df2['Carbon Peak Area'].values
    test_oxygen_peak_areas = test_peak_fitting_df2['Carbon Peak Area'].values

    # Feature scaling
    _training_carbon_portions = (training_carbon_portions-np.min(training_carbon_portions)) / (np.max(training_carbon_portions)-np.min(training_carbon_portions))
    _carbon_portions = (carbon_portions-np.min(training_carbon_portions)) / (np.max(training_carbon_portions)-np.min(training_carbon_portions))
    _test_carbon_portions = (test_carbon_portions-np.min(training_carbon_portions)) / (np.max(training_carbon_portions)-np.min(training_carbon_portions))

    _training_silicon_portions = (training_silicon_portions-np.min(training_silicon_portions)) / (np.max(training_silicon_portions)-np.min(training_silicon_portions))
    _silicon_portions = (silicon_portions-np.min(training_silicon_portions)) / (np.max(training_silicon_portions)-np.min(training_silicon_portions))
    _test_silicon_portions = (test_silicon_portions-np.min(training_silicon_portions)) / (np.max(training_silicon_portions)-np.min(training_silicon_portions))

    _training_oxygen_portions = (training_oxygen_portions-np.min(training_oxygen_portions)) / (np.max(training_oxygen_portions)-np.min(training_oxygen_portions))
    _oxygen_portions = (oxygen_portions-np.min(training_oxygen_portions)) / (np.max(training_oxygen_portions)-np.min(training_oxygen_portions))
    _test_oxygen_portions = (test_oxygen_portions-np.min(training_oxygen_portions)) / (np.max(training_oxygen_portions)-np.min(training_oxygen_portions))
    
    _training_carbon_peak_areas = (training_carbon_peak_areas-np.min(training_carbon_peak_areas)) / (np.max(training_carbon_peak_areas)-np.min(training_carbon_peak_areas))
    _carbon_peak_areas = (carbon_peak_areas-np.min(training_carbon_peak_areas)) / (np.max(training_carbon_peak_areas)-np.min(training_carbon_peak_areas))
    _test_carbon_peak_areas = (test_carbon_peak_areas-np.min(training_carbon_peak_areas)) / (np.max(training_carbon_peak_areas)-np.min(training_carbon_peak_areas))
    _training_silicon_peak_areas = (training_silicon_peak_areas-np.min(training_silicon_peak_areas)) / (np.max(training_silicon_peak_areas)-np.min(training_silicon_peak_areas))
    _silicon_peak_areas = (silicon_peak_areas-np.min(training_silicon_peak_areas)) / (np.max(training_silicon_peak_areas)-np.min(training_silicon_peak_areas))
    _test_silicon_peak_areas = (test_silicon_peak_areas-np.min(training_silicon_peak_areas)) / (np.max(training_silicon_peak_areas)-np.min(training_silicon_peak_areas))
    _training_oxygen_peak_areas = (training_oxygen_peak_areas-np.min(training_oxygen_peak_areas)) / (np.max(training_oxygen_peak_areas)-np.min(training_oxygen_peak_areas))
    _oxygen_peak_areas = (oxygen_peak_areas-np.min(training_oxygen_peak_areas)) / (np.max(training_oxygen_peak_areas)-np.min(training_oxygen_peak_areas))
    _test_oxygen_peak_areas = (test_oxygen_peak_areas-np.min(training_oxygen_peak_areas)) / (np.max(training_oxygen_peak_areas)-np.min(training_oxygen_peak_areas))

    result, analyze = activation_layer_multivar_linreg(np.array([
        _training_carbon_portions, _training_silicon_portions, _training_oxygen_portions,
        _training_carbon_peak_areas, _training_silicon_peak_areas, _training_oxygen_peak_areas
        ]).T, np.array(training_true_carbon))

    predicted_carbon = analyze(np.array([
        _carbon_portions, _silicon_portions, _oxygen_portions,
        _carbon_peak_areas, _silicon_peak_areas, _oxygen_peak_areas
        ]).T)
    predicted_training_carbon = analyze(np.array([
        _training_carbon_portions, _training_silicon_portions, _training_oxygen_portions,
        _training_carbon_peak_areas, _training_silicon_peak_areas, _training_oxygen_peak_areas
        ]).T)
    predicted_test_carbon = analyze(np.array([
        _test_carbon_portions, _test_silicon_portions, _test_oxygen_portions,
        _test_carbon_peak_areas, _test_silicon_peak_areas, _test_oxygen_peak_areas
        ]).T)

    mse_test = np.mean(np.abs(test_true_carbon - predicted_test_carbon))

    worst_accuracy = np.max(np.abs(test_true_carbon - predicted_test_carbon))

    info = {
        'mse': mse_test,
        'worst_accuracy': worst_accuracy,
        'fitting_df': fitting_df,
        'predicted_df': predicted_df,
        'combined_df': _combined_df,
        'train_cols': list(elem_exp_df.index),
        'carbon_portions': carbon_portions,
        'silicon_portions': silicon_portions,
        'oxygen_portions': oxygen_portions,
        'carbon_peak_areas': carbon_peak_areas,
        'silicon_peak_areas': silicon_peak_areas,
        'oxygen_peak_areas': oxygen_peak_areas,
        'training_carbon_portions': training_carbon_portions,
        'test_carbon_portions': test_carbon_portions,
        'true_carbon': true_carbon,
        'true_silicon': true_silicon,
        'true_oxygen': true_oxygen,
        'training_true_carbon': training_true_carbon,
        'test_true_carbon': test_true_carbon,
        'predicted_carbon': predicted_carbon,
        'predicted_training_carbon': predicted_training_carbon,
        'predicted_test_carbon': predicted_test_carbon,
        'method group': 'Compound Method',
        
    }
    return info

In [33]:
_material_sims = material_sims[material_sims['carbon_level']<=0.15]

In [34]:
_training_sims = _material_sims.sample(frac=0.7, random_state=42)
_training = DetectorReadings[_training_sims.index]
_test_sims = _material_sims.drop(_training_sims.index)
_test = DetectorReadings[_test_sims.index]

In [35]:
sims.loc[_training_sims.index, 'Training Set'] = True
sims.loc[_test_sims.index, 'Training Set'] = False
sims.to_excel('sims.xlsx')

In [36]:
from IPython.display import HTML
import base64

sims_file = 'sims.xlsx'
with open(sims_file, 'rb') as f:
    sims_b64 = base64.b64encode(f.read()).decode()

display(HTML(f"""
<a href="data:application/vnd.openxmlformats-officedocument.spreadsheetml.sheet;base64,{sims_b64}" 
   download="sims.xlsx" 
   style="display: inline-block; padding: 10px 20px; background-color: #4CAF50; color: white; text-decoration: none; border-radius: 4px; cursor: pointer;">
   Download sims.xlsx
</a>
"""))

In [37]:
DetectorReadings.to_excel('DetectorReadings.xlsx')

display(HTML(f"""
<a href="data:application/vnd.openxmlformats-officedocument.spreadsheetml.sheet;base64,{sims_b64}" 
   download="DetectorReadings.xlsx" 
   style="display: inline-block; padding: 10px 20px; background-color: #4CAF50; color: white; text-decoration: none; border-radius: 4px; cursor: pointer;">
   Download DetectorReadings.xlsx
</a>
"""))

# RUN

In [38]:

functions = {
    "Baseline and Peak Fitting - linear Baseline": PF_Linear,
    # "Baseline and Peak Fitting - Exponential Falloff Baseline": PF_Exp_Falloff,
    "Baseline and Peak Fitting - Exponential Falloff (C and Si)": PF_Exp_Falloff_Carbon_Silicon,
    # "Baseline and Peak Fitting - Exponential Falloff (C and O)": PF_Exp_Falloff_Carbon_Oxygen,
    "Baseline and Peak Fitting - Exponential Falloff (C only)": PF_Exp_Falloff_Carbon_Only,
    "Baseline and Peak Fitting - Exponential Falloff (C Si and O)": PF_Exp_Falloff_Full,
    "Baseline and Peak Fitting - Exponential Falloff (C Si and O) - DoubleGauss": PF_Exp_Falloff_Full_doublegauss,
    # "Component Analysis - Elemental Maps": CAelem,
    "Component Analysis - Elemental Maps (C Si and O) with Convex Minimizer": CAelemConvFull,
    "Component Analysis - Elemental Maps (C Si and O)": CAelemFull,
    "Component Analysis - Elemental Maps (C and Si)": CAelemCSi,
    "Component Analysis - Elemental Maps (C only)": CAelemC,
    "Baseline and Peak Fitting with Exponential Falloff + Elemental Maps": PF_Exp_Falloff_CAelem,
    "Baseline and Peak Fitting with Exponential Falloff + Elemental Maps with Linear Regression": PF_Exp_Falloff_CAelem_regression,
    "Baseline and Peak Fitting with Exponential Falloff + Elemental Maps with Feature Scaling": PF_Exp_Falloff_CAelem_featurescaling,
    "Baseline and Peak Fitting with Exponential Falloff + Elemental Maps with Linear Regression and Feature Scaling": PF_Exp_Falloff_CAelem_regression_feature_scaling,
    "Fully Featured Compound Method": FullFeatures,
    "Fully Featured Convex Endmembers": FullFeaturesConvex,
    "Component Analysis - Elemental Maps (C only)": CAelemC,
}

In [39]:
save = []
for name, func in functions.items():
    print(f"Running {name}...")
    # try:
    result = func(_training, _test, _training_sims[['carbon_level', 'silicon_level', 'oxygen_level']], _test_sims[['carbon_level', 'silicon_level', 'oxygen_level']])
    
    result['method'] = name
    save.append(result)
    # except Exception as e:
    #     print(f"Error running {name}: {e}")

Running Baseline and Peak Fitting - linear Baseline...
Running Baseline and Peak Fitting - Exponential Falloff (C and Si)...


/home/jac2462@uta.edu/Documents/.venv/lib/python3.10/site-packages/pandas/core/indexes/base.py:952: RuntimeWarning: overflow encountered in exp
  result = getattr(ufunc, method)(*new_inputs, **kwargs)
/home/jac2462@uta.edu/Documents/.venv/lib/python3.10/site-packages/scipy/optimize/_lsq/common.py:234: RuntimeWarning: overflow encountered in scalar divide
  ratio = actual_reduction / predicted_reduction


Running Baseline and Peak Fitting - Exponential Falloff (C only)...


/home/jac2462@uta.edu/Documents/.venv/lib/python3.10/site-packages/pandas/core/indexes/base.py:952: RuntimeWarning: overflow encountered in exp
  result = getattr(ufunc, method)(*new_inputs, **kwargs)
/home/jac2462@uta.edu/Documents/.venv/lib/python3.10/site-packages/scipy/optimize/_lsq/common.py:234: RuntimeWarning: overflow encountered in scalar divide
  ratio = actual_reduction / predicted_reduction


Running Baseline and Peak Fitting - Exponential Falloff (C Si and O)...


...

/home/jac2462@uta.edu/Documents/.venv/lib/python3.10/site-packages/pandas/core/indexes/base.py:952: RuntimeWarning: overflow encountered in exp
  result = getattr(ufunc, method)(*new_inputs, **kwargs)
/home/jac2462@uta.edu/Documents/.venv/lib/python3.10/site-packages/scipy/optimize/_lsq/common.py:234: RuntimeWarning: overflow encountered in scalar divide
  ratio = actual_reduction / predicted_reduction


Running Fully Featured Compound Method...


/home/jac2462@uta.edu/Documents/.venv/lib/python3.10/site-packages/pandas/core/indexes/base.py:952: RuntimeWarning: overflow encountered in exp
  result = getattr(ufunc, method)(*new_inputs, **kwargs)
/home/jac2462@uta.edu/Documents/.venv/lib/python3.10/site-packages/scipy/optimize/_lsq/common.py:234: RuntimeWarning: overflow encountered in scalar divide
  ratio = actual_reduction / predicted_reduction


Running Fully Featured Convex Endmembers...


/home/jac2462@uta.edu/Documents/.venv/lib/python3.10/site-packages/pandas/core/indexes/base.py:952: RuntimeWarning: overflow encountered in exp
  result = getattr(ufunc, method)(*new_inputs, **kwargs)
/home/jac2462@uta.edu/Documents/.venv/lib/python3.10/site-packages/scipy/optimize/_lsq/common.py:234: RuntimeWarning: overflow encountered in scalar divide
  ratio = actual_reduction / predicted_reduction


In [40]:

output = pd.DataFrame(save)#[['datasets used', 'carbon level', 'method', 'mse']]
output['id'] = output.index
# output



In [41]:
output.to_pickle('analysis_results.pkl')



In [42]:
output[['mse', 'method group', 'method', 'Variables']].to_excel('analysis_summary.xlsx', index=False)

In [43]:
display(HTML(f"""
<a href="data:application/vnd.openxmlformats-officedocument.spreadsheetml.sheet;base64,{sims_b64}" 
   download="analysis_summary.xlsx" 
   style="display: inline-block; padding: 10px 20px; background-color: #4CAF50; color: white; text-decoration: none; border-radius: 4px; cursor: pointer;">
   Download analysis_summary.xlsx
</a>
"""))